# DeepSeek-V4-Flash on Kaggle TPU v5e-8 (de-simplified engine)

Pure-JAX serving engine for the 284B-total / 13B-active FP4-expert MoE:
full CSA (ratio-4 overlapping compress + Lightning Indexer top-512) and
HCA (ratio-128 dense-over-compressed) attention, sliding window 128,
shared-KV MQA with attention sinks + inverse-RoPE outputs, mHC
hyper-connections, hash routing on layers 0-2, sqrtsoftplus noaux_tc
router, FP4 (e2m1 + e8m0 per-32) experts with hot banks per chip, MTP-1
speculative decode, 256k default context with compressed KV, and
server-side context auto-compaction.

- Weights: `deepseek-ai/DeepSeek-V4-Flash` (ungated, MIT).  With an
  HF_TOKEN the loader switches to the gated
  `orcarouter/DeepSeek-V4-Flash-Vision-Uncensored` variant — same text
  tensors; its 267 vision tensors (vision.*, aligner.*, image_*) are
  STRIPPED and never materialized.
- FP4 experts stay packed on host (~142 GB); decode banks hold fp4 bytes
  + e8m0 scales (halved bank traffic vs fp8), exact fixpoint refresh.
- API: OpenAI-compatible on :8080 + cloudflared tunnel; DSV4 thinks —
  streaming splits reasoning_content vs content; image/video requests
  get a clean 400.
- Validated on 8 simulated CPU devices: greedy determinism, starved-bank
  == full-bank exactness, MTP-1 lossless greedy, FP4 dequant exact,
  hash routing == tid2eid, compaction.

In [ ]:
import os, sys, time

print("checking TPU...")
import jax
print("jax", jax.__version__)
devs = jax.devices()
print("devices:", devs)
assert len(devs) == 8, f"expected 8 TPU devices, got {len(devs)}"
os.makedirs("/kaggle/tmp", exist_ok=True)


In [ ]:
# ---- engine modules: pull from GitHub first, embedded fallback ----
import os, sys, subprocess

# weights source: cebeuq 0731 abliterated (UNGATED, agent-tuned, DSpark-capable)
# is the default; deepseek-ai base as explicit fallback.  orcarouter only if
# explicitly requested via DSV4_REPO + HF_TOKEN (gated, vision-model base).
HF_TOKEN = None
try:
    from kaggle_web_client import UserSecretClient
    HF_TOKEN = UserSecretClient().get_secret("HF_TOKEN")
    print("HF token: loaded from Kaggle secret")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or None
    if HF_TOKEN:
        print("HF token: from env")

if not os.environ.get("DSV4_REPO"):
    os.environ["DSV4_REPO"] = "cebeuq/DeepSeek-V4-Flash-0731-abliterated"
# (explicit DSV4_REPO env always wins — e.g. orcarouter with HF_TOKEN,
#  or deepseek-ai/DeepSeek-V4-Flash as the ungated base fallback)
print("weights repo:", os.environ["DSV4_REPO"])

try:
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/lmaohhh/tpu-glm.git",
                    "/kaggle/tmp/tpu-glm"], check=True, timeout=120)
    sys.path.insert(0, "/kaggle/tmp/tpu-glm/src")
    print("engine: from GitHub @", subprocess.run(
        ["git", "-C", "/kaggle/tmp/tpu-glm", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True).stdout.strip())
except Exception as e:
    print("git pull failed -> using embedded modules:", e)

if "glmtpu.dsv4_runtime" not in sys.modules:
    MODS = {}
    MODS['dsv4_config.py'] = '''"""DeepSeek-V4-Flash serving config (de-simplified port).

Values from research/dsv4-0731-config.json + research/dsv4-port-spec.md §0
(all verified against the HF repo).  compress_ratios has 44 entries:
43 target layers + the MTP layer's attention type at index 43.
"""
from __future__ import annotations

import dataclasses
import json


@dataclasses.dataclass
class Dsv4Config:
    # ---- geometry (real values) ----
    hidden_size: int = 4096
    vocab_size: int = 129280
    n_layers: int = 43                 # target layers 0..42
    n_heads: int = 64
    head_dim: int = 512
    rope_head_dim: int = 64            # trailing dims of q/k/o
    q_lora_rank: int = 1024
    o_lora_rank: int = 1024
    o_groups: int = 8
    window_size: int = 128
    # indexer (CSA layers)
    index_n_heads: int = 64
    index_head_dim: int = 128
    index_topk: int = 512
    # moe
    n_experts: int = 256
    top_k: int = 6
    moe_inter: int = 2048
    routed_scaling_factor: float = 1.5
    swiglu_limit: float = 10.0
    n_hash_layers: int = 3             # layers 0-2: tid2eid hash routing
    # hyper-connections
    hc_mult: int = 4
    hc_sinkhorn_iters: int = 20
    hc_eps: float = 1e-6
    rms_norm_eps: float = 1e-6
    # rope
    rope_theta: float = 10000.0        # layers 0/1 + MTP (plain)
    compress_rope_theta: float = 160000.0  # CSA/HCA q/k + compressors
    yarn_factor: float = 16.0
    yarn_orig_ctx: int = 65536
    yarn_beta_fast: float = 32.0
    yarn_beta_slow: float = 1.0
    eos_ids: tuple = (1,)              # <｜end▁of▁sentence｜>
    # serving
    max_ctx: int = 262144              # 256k default (1M via config)
    prefill_chunk: int = 256           # multiple of every ratio (4,128)
    n_slots: int = 8                   # hot experts per chip per layer

    compress_ratios: tuple = ()        # 44 entries; set by real()/tiny()

    @staticmethod
    def real(max_ctx: int = 262144) -> "Dsv4Config":
        c = Dsv4Config(max_ctx=max_ctx)
        ratios = [0, 0]
        for _ in range(21):
            ratios += [4, 128]
        ratios += [4, 0]               # layer 42 CSA; index 43 = MTP sliding
        c.compress_ratios = tuple(ratios)
        assert len(c.compress_ratios) == 44
        assert c.prefill_chunk % 128 == 0
        return c

    @staticmethod
    def tiny() -> "Dsv4Config":
        c = Dsv4Config(
            hidden_size=128, vocab_size=512, n_layers=6,
            n_heads=8, head_dim=80, rope_head_dim=16,
            q_lora_rank=32, o_lora_rank=32, o_groups=8,
            window_size=8,
            index_n_heads=8, index_head_dim=32, index_topk=4,
            n_experts=16, top_k=2, moe_inter=32,
            hc_mult=2, hc_sinkhorn_iters=3,
            max_ctx=256, prefill_chunk=16, n_slots=2,
        )
        c.compress_ratios = (0, 0, 4, 8, 4, 8, 0)
        return c

    # ------------------------------------------------------------ helpers
    def ratio(self, l: int) -> int:
        return self.compress_ratios[l]

    def layer_type(self, l: int) -> str:
        r = self.ratio(l)
        return {0: "sliding", 4: "csa", 128: "hca"}[r if r in (0, 4, 128)
                                                   else _tiny_ratio_map(r)]

    def is_hash(self, l: int) -> bool:
        return l < self.n_hash_layers

    def n_comp(self, ratio: int) -> int:
        """Compressed-cache entries per compressed layer (static size)."""
        return self.max_ctx // ratio

    def to_json(self) -> str:
        d = dataclasses.asdict(self)
        d["eos_ids"] = list(self.eos_ids)
        d["compress_ratios"] = list(self.compress_ratios)
        return json.dumps(d, indent=1)

    @staticmethod
    def from_json(s: str) -> "Dsv4Config":
        d = json.loads(s)
        d["eos_ids"] = tuple(d["eos_ids"])
        d["compress_ratios"] = tuple(d["compress_ratios"])
        return Dsv4Config(**d)


def _tiny_ratio_map(r: int) -> int:
    # tiny config uses ratio 8 as the HCA stand-in
    return 128 if r == 8 else r
'''
    MODS['dsv4_fp4.py'] = '''"""FP4 (e2m1) expert format: packing, dequant, simulation, reference decoder.

Layout (research/dsv4-port-spec.md §3, from DeepSeek inference/convert.py +
kernel.py):
  - e2m1: 1 sign + 2 exp + 1 mantissa; 16 values
    [0, .5, 1, 1.5, 2, 3, 4, 6] and their negatives.
  - packed 2 per byte along K, LOW nibble = earlier K element.
  - scale: e8m0 (power-of-2 exponent byte) per 32 fp4 elements along K,
    per output row: scale tensor [out, in//32] uint8.
Dequant in f32 is EXACT (e2m1 mantissa 2 bits, e8m0 power-of-2); bf16 cast
lossless too (values ±{0,.5,1,1.5,2,3,4,6}·2^k fit in 8-bit mantissa).
"""
from __future__ import annotations

import numpy as np

FP4_TABLE = np.array(
    [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0,
     0.0, -0.5, -1.0, -1.5, -2.0, -3.0, -4.0, -6.0], np.float32)

FP4_BLOCK = 32


# ------------------------------------------------------------------ numpy
def dequant_np(packed_u8: np.ndarray, scale_u8: np.ndarray) -> np.ndarray:
    """Reference decoder.  packed [out, in//2] uint8, scale [out, in//32]
    uint8 (e8m0) -> f32 [out, in].  Low nibble first along K."""
    out_dim, half_in = packed_u8.shape
    in_dim = half_in * 2
    low = (packed_u8 & 0x0F).astype(np.int64)
    high = (packed_u8 >> 4).astype(np.int64)
    vals = np.empty((out_dim, in_dim), np.float32)
    vals[:, 0::2] = FP4_TABLE[low]
    vals[:, 1::2] = FP4_TABLE[high]
    # e8m0 scale: value = 2^(byte - 127)
    scales = np.exp2(scale_u8.astype(np.float32) - 127.0)  # [out, in//32]
    nblocks = in_dim // FP4_BLOCK
    for j in range(nblocks):
        vals[:, j * FP4_BLOCK:(j + 1) * FP4_BLOCK] *= scales[:, j:j + 1]
    return vals


def quantize_np(w: np.ndarray) -> tuple:
    """Quantize f32 [out, in] -> (packed u8 [out, in//2], scale u8
    [out, in//32]) with per-32 rowwise e8m0 scales (round-to-nearest e2m1).
    Used by dsv4_params to fabricate fp4 test weights."""
    out_dim, in_dim = w.shape
    assert in_dim % FP4_BLOCK == 0 and in_dim % 2 == 0
    blocks = w.reshape(out_dim, in_dim // FP4_BLOCK, FP4_BLOCK)
    amax = np.maximum(np.abs(blocks).max(axis=-1), 1e-30)  # [out, nb]
    # e8m0 scale = 2^ceil(log2(amax / 6)) (power-of-2, like the kernel)
    e = np.ceil(np.log2(amax / 6.0))
    e = np.clip(e, -127, 127)
    scale_u8 = (e + 127.0).astype(np.uint8)
    s = np.exp2(e)
    t = blocks / s[..., None]                     # target range [-6, 6]
    # round to nearest e2m1 magnitude: pick nibble index 0..7
    grid = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0], np.float32)
    a = np.abs(t)
    idx = np.clip(np.searchsorted(grid, a), 1, 7)      # candidate upper
    left, right = grid[idx - 1], grid[idx]
    up = (a - left) > (right - a)                       # round up?
    nib7 = np.where(up, idx, idx - 1)                   # magnitude idx 0..7
    nib7 = np.where(a == 0.0, 0, nib7)
    # e2m1 nibble: magnitude idx | 0x8 if negative
    nib = (nib7 + np.where(t < 0, 8, 0)).astype(np.int64)
    nib = nib.reshape(out_dim, in_dim)
    packed = (nib[:, 0::2] | (nib[:, 1::2] << 4)).astype(np.uint8)
    return packed, scale_u8


# ------------------------------------------------------------------ jax
def dequant_jax(packed, scale):
    """In-graph dequant.  packed [out, in//2] uint8, scale [out, in//32]
    uint8 -> bf16 [out, in].  Nibble ops + 16-entry LUT + exp2(scale-127)."""
    import jax.numpy as jnp

    out_dim, half_in = packed.shape
    in_dim = half_in * 2
    lut = jnp.asarray(FP4_TABLE, jnp.float32)          # [16]
    low = (packed & 0x0F).astype(jnp.int32)            # [out, in//2]
    high = (packed >> 4).astype(jnp.int32)
    v0 = lut[low]                                       # even K elems
    v1 = lut[high]                                      # odd K elems
    vals = jnp.empty((out_dim, in_dim), jnp.float32)
    vals = vals.at[:, 0::2].set(v0).at[:, 1::2].set(v1)
    s = jnp.exp2(scale.astype(jnp.float32) - 127.0)     # [out, in//32]
    nb = in_dim // FP4_BLOCK
    cols = jnp.arange(in_dim) // FP4_BLOCK
    vals = vals * s[:, cols]
    return vals.astype(jnp.bfloat16)


def fp4_sim_jax(x, block: int = FP4_BLOCK):
    """FP4 activation simulation (indexer q/k): quant-dequant with per-32
    e8m0 scales, round-to-nearest e2m1 grid.  x [..., N] f32 -> f32."""
    import jax
    import jax.numpy as jnp

    shape = x.shape
    n = shape[-1]
    assert n % block == 0
    xb = x.reshape(-1, n // block, block)
    amax = jnp.maximum(jnp.abs(xb).max(axis=-1, keepdims=True), 1e-30)
    e = jnp.ceil(jnp.log2(amax / 6.0))
    s = jnp.exp2(e)
    t = xb / s
    grid = jnp.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0], jnp.float32)
    a = jnp.abs(t)
    idx = jnp.clip(jnp.searchsorted(grid, a), 1, 7)
    left, right = grid[idx - 1], grid[idx]
    chosen = jnp.where(a - left <= right - a, left, right)
    mag = jnp.where(a == 0.0, 0.0, chosen)
    y = jnp.sign(t) * mag * s
    return y.reshape(shape)
'''
    MODS['dsv4_layers.py'] = '''"""DeepSeek-V4-Flash per-layer math in pure JAX (de-simplified port).

Mirrors research/dsv4-inference-model.py + kernel.py (line-level where
noted) and research/dsv4-port-spec.md §1/§4.  Every function runs on ONE
chip with LOCAL shards inside jax.pmap(axis_name='tp').

Per-chip sharding (d = 8 chips):
  attention   64 heads -> 8 local heads/chip == 1 wo_a group (o_groups 8):
              wq_b/wo_a/wo_b/attn_sink head/group-sliced; wq_a/wkv/norms
              replicated; compressed caches replicated (MQA: every chip
              needs all KV); final wo_b partial -> psum.
  indexer     heads 64 -> 8/chip (scores psum over heads); weights_proj
              head-sliced; its compressor replicated.
  MoE         shared expert w1/w3 row-sharded, w2 col-sharded + psum;
              routed experts in per-chip FP4 banks (fixpoint refresh);
              router + tid2eid replicated.

KV cache formats (per layer):
  ring        [B, W, 512]: nope 448 dims fp8 u8 + per-64 e8m0-ish scales
              stored separately, rope 64 dims bf16.  Kept as one u8 array
              [B, W, 448] + scales [B, W, 7] + rope [B, W, 64].
  compressed  same split, [B, max_ctx//ratio, ...].
  indexer     keys-only bf16 [B, max_ctx//ratio, 128].
"""
from __future__ import annotations

import jax
import jax.numpy as jnp
from jax import lax

from .dsv4_fp4 import dequant_jax
from .fp8 import F8_LUT

F8 = jnp.asarray(F8_LUT, jnp.float32)      # [256] e4m3 LUT


# ===========================================================================
# basic ops
# ===========================================================================

def rms_norm(x, w, eps):
    x32 = x.astype(jnp.float32)
    x32 = x32 * lax.rsqrt(jnp.mean(x32 * x32, axis=-1, keepdims=True) + eps)
    return (w.astype(jnp.float32) * x32).astype(x.dtype)


def rms_norm_no_w(x, eps):
    x32 = x.astype(jnp.float32)
    return (x32 * lax.rsqrt(jnp.mean(x32 * x32, axis=-1, keepdims=True)
                            + eps)).astype(x.dtype)


def swiglu_clamped(gate, up, limit):
    gate = jnp.clip(gate, None, limit)
    up = jnp.clip(up, -limit, limit)
    return (jax.nn.silu(gate.astype(jnp.float32)).astype(gate.dtype)) * up


# ===========================================================================
# rope (dual tables, in-graph from static freq vectors)
# ===========================================================================

def yarn_freqs(cfg, compress: bool) -> jnp.ndarray:
    """Static [rd/2] frequency vector (YaRN interpolation for compress)."""
    rd = cfg.rope_head_dim
    base = cfg.compress_rope_theta if compress else cfg.rope_theta
    freqs = 1.0 / (base ** (jnp.arange(0, rd, 2, dtype=jnp.float32) / rd))
    if compress and cfg.yarn_orig_ctx > 0:
        # find_correction_range (model.py precompute_freqs_cis)
        def corr_dim(rot):
            return (rd * jnp.log(cfg.yarn_orig_ctx /
                                 (rot * 2 * jnp.pi)) / (2 * jnp.log(base)))
        low = jnp.floor(corr_dim(cfg.yarn_beta_fast)).astype(jnp.int32)
        high = jnp.ceil(corr_dim(cfg.yarn_beta_slow)).astype(jnp.int32)
        low, high = jnp.maximum(low, 0), jnp.minimum(high, rd - 1)
        ramp = jnp.clip((jnp.arange(rd // 2, dtype=jnp.float32) - low)
                        / jnp.maximum(high - low, 1), 0.0, 1.0)
        smooth = 1.0 - ramp
        freqs = freqs / cfg.yarn_factor * (1.0 - smooth) + freqs * smooth
    return freqs                                  # [rd/2] f32 static


def apply_rope(x, pos, freqs, inverse=False):
    """x [B, S, ..., head] (positions on axis 1); pos [S] i32; freqs
    [rd/2] static f32.  Interleaved-pair rope on the TRAILING rd dims
    of the last axis (consecutive channel pairs, model.py
    view_as_complex).  Returns rotated copy."""
    rd2 = freqs.shape[0]
    rd = rd2 * 2
    ang = pos[:, None].astype(jnp.float32) * freqs[None, :]   # [S, rd/2]
    cos = jnp.cos(ang)
    sin = jnp.sin(ang)
    # broadcast over axis 1 (S): [1, S, 1, ..., rd/2] matching x.ndim
    bshape = [1, x.shape[1]] + [1] * (x.ndim - 3) + [rd2]
    cosb = cos.reshape(bshape)
    sinb = sin.reshape(bshape)
    even = x[..., -rd::2].astype(jnp.float32)
    odd = x[..., -rd + 1::2].astype(jnp.float32)
    if inverse:                                   # conj (de-rotation)
        re, im = even * cosb + odd * sinb, odd * cosb - even * sinb
    else:
        re, im = even * cosb - odd * sinb, even * sinb + odd * cosb
    # interleave (re, im) pairs back onto the trailing rd channels:
    # even/odds are [..., rd/2]; build [..., rd] via stack + reshape
    rot = jnp.stack([re, im], axis=-1)            # [..., rd/2, 2]
    rot = rot.reshape(tuple(re.shape[:-1]) + (rd,))
    if rd == x.shape[-1]:
        return rot.astype(x.dtype)
    out = jnp.concatenate([x[..., :-rd].astype(jnp.float32), rot], axis=-1)
    return out.astype(x.dtype)


def hadamard(x):
    """Randomized-Hadamard-style rotation used pre-FP4 in the indexer.
    Deterministic n×n Hadamard (power-of-2 dims), scale n^-0.5."""
    n = x.shape[-1]
    assert n & (n - 1) == 0, "hadamard needs pow2 last dim"
    # build H_n by Kronecker doubling (static, tiny for 128/32)
    h = jnp.array([[1.0, 1.0], [1.0, -1.0]], jnp.float32)
    k = 1
    while (1 << k) < n:
        h = jnp.concatenate([jnp.concatenate([h, h], axis=1),
                             jnp.concatenate([h, -h], axis=1)], axis=0)
        k += 1
    return (x.astype(jnp.float32) @ (h * (n ** -0.5))).astype(x.dtype)


# ===========================================================================
# fp8-sim KV quantization (block 64, ue8m0 power-of-2 scales, inplace)
# ===========================================================================

def kv_fp8_sim(x64):
    """x64 [..., N] (N%64==0) -> quant-dequant f32 (matches kernel
    act_quant inplace, scale_fmt=ue8m0: s = 2^ceil(log2(amax/448)))."""
    shp = x64.shape
    nb = shp[-1] // 64
    xb = x64.astype(jnp.float32).reshape(*shp[:-1], nb, 64)
    amax = jnp.maximum(jnp.abs(xb).max(axis=-1, keepdims=True), 1e-4)
    e = jnp.ceil(jnp.log2(amax / 448.0))
    s = jnp.exp2(e)
    q = jnp.clip(xb / s, -448.0, 448.0)
    # round to e4m3 grid via LUT on nearest-byte: use bitcast round-trip
    q8 = q.astype(jnp.float8_e4m3fn).astype(jnp.float32)
    return (q8 * s).reshape(shp)


def kv_pack(x, rd):
    """x [..., dh] f32 -> (nope u8 [..., dh-rd] fp8 codes (64-blocks),
    scales f32 [..., (dh-rd)//64], rope bf16 [..., rd])."""
    shp = x.shape
    nope = x[..., :-rd]
    rope = x[..., -rd:].astype(jnp.bfloat16)
    nb = (shp[-1] - rd) // 64
    xb = nope.reshape(*shp[:-1], nb, 64)
    amax = jnp.maximum(jnp.abs(xb).max(axis=-1, keepdims=True), 1e-4)
    e = jnp.ceil(jnp.log2(amax / 448.0))
    s = jnp.exp2(e)
    q = jnp.clip(xb / s, -448.0, 448.0)
    b = q.astype(jnp.float8_e4m3fn)
    u8 = lax.bitcast_convert_type(b, jnp.uint8).reshape(*shp[:-1],
                                                        shp[-1] - rd)
    scales = s.reshape(*shp[:-1], nb)
    return u8, scales, rope


def kv_unpack(u8, scales, rope):
    """inverse of kv_pack -> [..., dh] bf16."""
    codes = lax.bitcast_convert_type(
        u8, jnp.float8_e4m3fn).astype(jnp.float32)
    shp = codes.shape
    nope = (codes.reshape(*shp[:-1], -1, 64)
            * scales[..., None]).reshape(shp)
    return jnp.concatenate(
        [nope.astype(jnp.bfloat16), rope], axis=-1)


# ===========================================================================
# mHC hyper-connections (model.py hc_pre/hc_post, kernel hc_split_sinkhorn)
# ===========================================================================

def hc_site(hc, streams, cfg):
    """hc = {"fn": [mix, H*D] f32, "base": [mix] f32, "scale": [3] f32}.
    streams [B,S,H,D] -> (post [B,S,H], comb [B,S,H,H] indexed [b,s,j,k],
    collapsed [B,S,D]).  comb consumed TRANSPOSED downstream."""
    B, S, H, D = streams.shape
    flat = streams.reshape(B, S, H * D).astype(jnp.float32)
    flat = rms_norm_no_w(flat, cfg.rms_norm_eps)
    mixes = flat @ hc["fn"].astype(jnp.float32).T       # [B,S,(2+H)H]
    pre_w, post_w, comb_w = jnp.split(mixes, [H, 2 * H], axis=-1)
    pre_b, post_b, comb_b = jnp.split(hc["base"].astype(jnp.float32),
                                      [H, 2 * H])
    pre_s, post_s, comb_s = hc["scale"][0], hc["scale"][1], hc["scale"][2]
    pre = jax.nn.sigmoid(pre_w * pre_s + pre_b) + cfg.hc_eps
    post = 2.0 * jax.nn.sigmoid(post_w * post_s + post_b)
    cl = comb_w.reshape(B, S, H, H) * comb_s + comb_b.reshape(H, H)
    comb = jax.nn.softmax(cl, axis=-1) + cfg.hc_eps
    comb = comb / (jnp.sum(comb, axis=-2, keepdims=True) + cfg.hc_eps)
    for _ in range(cfg.hc_sinkhorn_iters - 1):
        comb = comb / (jnp.sum(comb, axis=-1, keepdims=True) + cfg.hc_eps)
        comb = comb / (jnp.sum(comb, axis=-2, keepdims=True) + cfg.hc_eps)
    collapsed = jnp.sum(pre[..., None] * streams, axis=2)
    return post, comb, collapsed.astype(streams.dtype)


def hc_apply(post, comb, sub_out, streams):
    """streams'[k] = post[k]*sub_out + Σ_j comb[j,k]*stream[j]."""
    term_a = post[..., None].astype(sub_out.dtype) * sub_out[..., None, :]
    term_b = jnp.einsum("bsjk,bsjd->bskd", comb.astype(streams.dtype),
                        streams)
    return term_a + term_b


# ===========================================================================
# attention core (shared by sliding / CSA / HCA)
# ===========================================================================

def sparse_attn_core(q, kv_ctx, kv_valid, sink, scale):
    """q [B,S,Hl,dh]; kv_ctx [B,N,dh] bf16 (dequantized); kv_valid
    [B,S,N] f32; sink [Hl] f32.  Softmax with the per-head sink added
    to the denominator (kernel sparse_attn).  Returns o [B,S,Hl,dh]."""
    scores = jnp.einsum("bshd,bnd->bhsn",
                        q.astype(jnp.float32),
                        kv_ctx.astype(jnp.float32)) * scale   # [B,Hl,S,N]
    scores = scores.astype(jnp.float32)
    neg = -1e30
    mask = kv_valid[:, None, :, :] > 0.5                # [B,1,S,N]
    scores = jnp.where(mask, scores, neg)
    m = jnp.max(scores, axis=-1, keepdims=True)
    p = jnp.exp(scores - m)
    p = jnp.where(mask, p, 0.0)
    sum_p = jnp.sum(p, axis=-1, keepdims=True)
    sum_p = sum_p + jnp.exp(sink[None, :, None, None] - m)   # sink
    o = jnp.einsum("bhsn,bnd->bshd", p, kv_ctx.astype(jnp.float32))
    o = o / jnp.transpose(sum_p, (0, 2, 1, 3))       # [B,S,Hl,1]
    return o.astype(jnp.bfloat16)


def grouped_o_proj(o, wo_a_blk, wo_b_blk):
    """o [B,S,Hl,dh] -> group flatten [B,S,Hl*dh] (Hl*dh == H*dh/G_local
    only when G_local groups per chip; here 1 group/chip) ->
    wo_a_blk [olr, Hl*dh] -> [B,S,olr] -> wo_b_blk [D, olr] partial."""
    B, S, Hl, dh = o.shape
    flat = o.reshape(B, S, Hl * dh)
    mid = flat @ wo_a_blk.astype(jnp.bfloat16).T          # [B,S,olr]
    return mid @ wo_b_blk.astype(jnp.bfloat16).T          # partial [B,S,D]


# ===========================================================================
# compressor (CSA ratio-4 overlap + HCA ratio-128 non-overlap)
# ===========================================================================

def compressor_proj(p, x):
    """fp32 projections (checkpoint stores bf16; reference computes fp32).
    x [B,S,D] -> (kv [B,S,coff*Dc] f32, score [B,S,coff*Dc] f32)."""
    x32 = x.astype(jnp.float32)
    kv = x32 @ p["wkv"].astype(jnp.float32).T
    sc = x32 @ p["wgate"].astype(jnp.float32).T
    return kv, sc


def compressor_ape8(p):
    """ape [ratio, coff*Dc] -> transformed-layout bias table
    [coff*ratio, Dc]: slot k<ratio -> ape[k, :Dc] (Ca cols),
    slot k>=ratio -> ape[k-ratio, Dc:] (Cb cols).  Non-overlap
    (coff=1): ape [ratio, Dc] as-is."""
    ape = p["ape"].astype(jnp.float32)         # [ratio, coff*Dc]
    ratio = ape.shape[0]
    coff = ape.shape[1] // p["norm"].shape[0]
    d = p["norm"].shape[0]
    if coff == 1:
        return ape
    return jnp.concatenate([ape[:, :d], ape[:, d:]], axis=0)  # [2r, d]


def compressor_prefill(p, x, state, pos0, cfg):
    """Vectorized over the chunk's windows.  Requires pos0 % ratio == 0 and
    S % ratio == 0 (runtime guarantees; pos0 traced or static).
    Returns (entries [B, S/ratio, Dc], new_state)."""
    ratio = p["ape"].shape[0]
    coff = p["ape"].shape[1] // p["norm"].shape[0]
    d = p["norm"].shape[0]
    kv, sc = compressor_proj(p, x)                   # [B,S,coff*d]
    B, S = kv.shape[0], kv.shape[1]
    nw = S // ratio
    kvw = kv.reshape(B, nw, ratio, coff * d)         # window-major
    scw = sc.reshape(B, nw, ratio, coff * d)
    ape = p["ape"].astype(jnp.float32)               # [ratio, coff*d]
    scw = scw + ape[None, None]                      # in-window ape bias
    if coff == 2:
        # overlap: entry i pools [Ca(win i-1) cols :d | Cb(win i) cols d:]
        ca_prev = state[0][:, None, :ratio, :d]      # [B,1,r,d]
        sc_prev = state[1][:, None, :ratio, :d]
        ca_kv = jnp.concatenate(
            [ca_prev, kvw[:, :-1, :, :d]], axis=1)   # [B,nw,r,d]
        ca_sc = jnp.concatenate(
            [sc_prev, scw[:, :-1, :, :d]], axis=1)
        cb_kv = kvw[:, :, :, d:]                     # [B,nw,ratio,d]
        cb_sc = scw[:, :, :, d:]
        slots_kv = jnp.concatenate([ca_kv, cb_kv], axis=2)   # [B,nw,2r,d]
        slots_sc = jnp.concatenate([ca_sc, cb_sc], axis=2)
    else:
        slots_kv = kvw                               # [B,nw,ratio,d]
        slots_sc = scw
    wts = jax.nn.softmax(slots_sc, axis=2)           # over slots
    entry = jnp.sum(slots_kv * wts, axis=2)          # [B,nw,d]
    entry = rms_norm(entry, p["norm"].astype(jnp.float32), cfg.rms_norm_eps)
    # new state: last window full projection (kv/score incl. ape)
    new_kv_state = jnp.concatenate(
        [kvw[:, -1], jnp.zeros((B, ratio, coff * d), jnp.float32)], axis=1)
    new_sc_state = jnp.concatenate(
        [scw[:, -1], jnp.zeros((B, ratio, coff * d), jnp.float32)], axis=1)
    return entry, (new_kv_state, new_sc_state)


def compressor_decode(p, x, state, pos, cfg):
    """Single token at global position pos.  state as above.  Returns
    (entry [B,1,Dc] or None, new_state).  Mirrors model.py Compressor
    decode branch."""
    ratio = p["ape"].shape[0]
    coff = p["ape"].shape[1] // p["norm"].shape[0]
    d = p["norm"].shape[0]
    kv, sc = compressor_proj(p, x)                   # [B,1,coff*d]
    ape = p["ape"].astype(jnp.float32)
    slot_pos = pos % ratio                           # traced i32
    sc = sc + ape[slot_pos][None, None]              # [B,1,coff*d]
    kv_state, sc_state = state
    B = kv.shape[0]
    slot = (ratio + pos % ratio) if coff == 2 else (pos % ratio)
    onehot = (jnp.arange(kv_state.shape[1]) == slot)          # [rows]
    kv_state = jnp.where(onehot[None, :, None],
                         jnp.broadcast_to(kv, (B, 1, coff * d)), kv_state)
    sc_state = jnp.where(onehot[None, :, None],
                         jnp.broadcast_to(sc, (B, 1, coff * d)), sc_state)
    should = ((pos + 1) % ratio == 0).astype(jnp.float32)
    entry = jnp.zeros((B, 1, d), jnp.float32)
    if coff == 2:
        pool_kv = jnp.concatenate(
            [kv_state[:, :ratio, :d], kv_state[:, ratio:, d:]], axis=1)
        pool_sc = jnp.concatenate(
            [sc_state[:, :ratio, :d], sc_state[:, ratio:, d:]], axis=1)
    else:
        pool_kv, pool_sc = kv_state, sc_state
    wts = jax.nn.softmax(pool_sc, axis=1)
    e = jnp.sum(pool_kv * wts, axis=1, keepdims=True)     # [B,1,d]
    e = rms_norm(e, p["norm"].astype(jnp.float32), cfg.rms_norm_eps)
    entry = should * e
    # rotate state when a window completed
    if coff == 2:
        new_kv = jnp.where(should > 0.5,
                           jnp.concatenate([kv_state[:, ratio:],
                                            jnp.zeros_like(kv_state[:, :ratio])],
                                           axis=1), kv_state)
        new_sc = jnp.where(should > 0.5,
                           jnp.concatenate([sc_state[:, ratio:],
                                            jnp.zeros_like(sc_state[:, :ratio])],
                                           axis=1), sc_state)
    else:
        new_kv, new_sc = kv_state, sc_state
    return entry, should, (new_kv, new_sc)


# ===========================================================================
# indexer (CSA layers)
# ===========================================================================

def indexer_scores(p_idx, qr, x, idx_keys, pos0, cfg):
    """qr [B,S,qlora] (shared q-lora latent, post q_norm); x [B,S,D];
    idx_keys [B, Ci, 128] bf16 (compressed indexer cache).  Returns
    scores [B,S,Ci] f32 with -inf at invalid (causal / unwritten)."""
    rd = cfg.rope_head_dim
    freqs = p_idx["_cfreqs"]                          # static [rd/2]
    q = qr @ p_idx["wq_b"].astype(jnp.bfloat16).T     # [B,S,Hl*128]
    Hl = p_idx["wq_b"].shape[0] // cfg.index_head_dim
    q = q.reshape(qr.shape[0], qr.shape[1], Hl, cfg.index_head_dim)
    pos = pos0 + jnp.arange(qr.shape[1])
    q = apply_rope(q, pos, freqs)
    q = hadamard(q.reshape(*q.shape[:-1], -1)).reshape(q.shape)
    from .dsv4_fp4 import fp4_sim_jax
    q = fp4_sim_jax(q.astype(jnp.float32), 32).astype(jnp.bfloat16)
    w = (x @ p_idx["weights_proj"].astype(jnp.bfloat16).T) \
        * (cfg.index_head_dim ** -0.5 * Hl ** -0.5)  # hmm: n_heads total
    # NOTE: softmax_scale uses TOTAL heads for the 64^-0.5 factor; with
    # head sharding we use local count and psum below.
    # score_e = Σ_h w_h ReLU(q_h · k_e)
    s = jnp.einsum("bshd,btd->bsht",
                   q.astype(jnp.float32), idx_keys.astype(jnp.float32))
    s = jnp.maximum(s, 0.0)                           # ReLU
    s = jnp.einsum("bsht,bsh->bst", s, w.astype(jnp.float32))
    return s, pos


# ===========================================================================
# MoE (hash routing + noaux_tc sqrtsoftplus + FP4 banks + shared expert)
# ===========================================================================

def moe_router(p_gate, h, input_ids, cfg):
    """p_gate = {"w": [E,D], "bias": [E] or None, "tid2eid": [V,K] or None}.
    h [B,S,D]; input_ids [B,S] i32.  Returns (weights [B,S,K] f32,
    ids [B,S,K] i32) — model.py Gate."""
    logits = h.astype(jnp.float32) @ p_gate["w"].astype(jnp.float32).T
    scores = jnp.sqrt(jax.nn.softplus(logits))        # sqrtsoftplus
    if p_gate.get("tid2eid") is not None:
        ids = p_gate["tid2eid"][input_ids]            # [B,S,K] hash lookup
    else:
        choice = scores + p_gate["bias"].astype(jnp.float32)
        ids = lax.top_k(choice, cfg.top_k)[1]
    weights = jnp.take_along_axis(scores, ids, axis=-1)
    weights = weights / (jnp.sum(weights, axis=-1, keepdims=True) + 1e-20)
    weights = weights * cfg.routed_scaling_factor
    return weights, ids


def fp4_bank_core(bank, h, w, ids, cfg):
    """Per-chip FP4 expert bank.  bank: {"w1": [n,I,D/2] u8,
    "w1_s": [n,I,D/32] u8, "w2"/"w2_s", "w3"/"w3_s", "ids": [n] i32}.
    Every chip applies resident experts to ALL tokens; psum makes it
    exact given host coverage."""
    B, S, D = h.shape
    n = bank["ids"].shape[0]
    out = jnp.zeros((B, S, D), jnp.float32)
    for s_i in range(n):
        e = bank["ids"][s_i]
        match = (ids == e).astype(jnp.float32)         # [B,S,K]
        coeff = jnp.sum(match * w, axis=-1)            # [B,S]
        hb = h.astype(jnp.bfloat16)
        w1 = dequant_jax(bank["w1"][s_i], bank["w1_s"][s_i])   # [I,D]
        w3 = dequant_jax(bank["w3"][s_i], bank["w3_s"][s_i])
        w2 = dequant_jax(bank["w2"][s_i], bank["w2_s"][s_i])   # [D,I]
        gate = hb @ w1.T                                 # [B,S,I]
        up = hb @ w3.T
        hid = swiglu_clamped(gate, up, cfg.swiglu_limit)
        y = hid @ w2.T                                   # [B,S,D]
        out = out + y.astype(jnp.float32) * coeff[..., None]
    return lax.psum(out.astype(h.dtype), "tp")


def dense_mlp_core(p, h, cfg):
    """Shared expert (BF16 after host dequant): w1/w3 row-sharded, w2
    col-sharded + psum.  p = {"w1": [I/d, D], "w3": ..., "w2": [D, I/d]}."""
    gate = h @ p["w1"].T
    up = h @ p["w3"].T
    hid = swiglu_clamped(gate, up, cfg.swiglu_limit)
    return lax.psum(hid @ p["w2"].T, "tp")
'''
    MODS['dsv4_params.py'] = '''"""Tiny-config fake weights for the DSV4 engine (structural tests).

Produces params_by_chip matching dsv4_loader's real-weight layout:

params_by_chip[c] = {
  l: {
    "attn_hc": {"fn","base","scale"},          # replicated f32
    "ffn_hc": {...},
    "attn_norm": [D], "ffn_norm": [D],
    "wq_a": [ql,D] bf16, "q_norm": [ql],
    "wq_b": [H/d*dh, ql] bf16 (head-sharded),
    "wkv": [dh, D] bf16, "kv_norm": [dh],
    "wo_a": [olr, H/d*dh] bf16, "wo_b": [D, olr] bf16 (group-sharded),
    "attn_sink": [H/d] f32,
    "comp": {...} | None,                       # compressor (replicated)
    "idx": {"wq_b","weights_proj"} | None,      # indexer (head-sharded)
    "icomp": {...} | None,                      # indexer compressor
    "gate": {"w","bias"|"tid2eid"},             # router (replicated)
    "shared": {"w1","w3","w2"},                 # shared expert (sharded)
  },
  "mtp": {attn (sliding) + ffn + e/h_proj + norms + hc_head_*},
  "head": {"fn","base","scale"},                # global hc_head
  "final_ln": [D],
}

Also returns (embed [V,D] f32, lm_head [V,D] f32,
expert_host[(key, e)] = {"w1","w1_s","w3","w3_s","w2","w2_s"} with
key = layer int or "mtp"; FP4 packed u8 + e8m0 u8 scales).
"""
from __future__ import annotations

import numpy as np

from .dsv4_config import Dsv4Config
from .dsv4_fp4 import quantize_np


def _bf(rng, shape, scale=0.02):
    a = (rng.standard_normal(shape) * scale).astype(np.float32)
    import jax.numpy as jnp
    return np.asarray(jnp.asarray(a).astype(jnp.bfloat16).astype(np.float32))


def _f32(rng, shape, scale=0.02):
    return (rng.standard_normal(shape) * scale).astype(np.float32)


def make_fake(cfg: Dsv4Config, d: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    D = cfg.hidden_size
    Hc = cfg.hc_mult
    mix = (2 + Hc) * Hc
    dh, rd = cfg.head_dim, cfg.rope_head_dim
    Hl = cfg.n_heads // d                      # local heads (1 group/chip)
    ql, ol = cfg.q_lora_rank, cfg.o_lora_rank
    E, I = cfg.n_experts, cfg.moe_inter

    def hc():
        return {"fn": _f32(rng, (mix, Hc * D)),
                "base": _f32(rng, (mix,), 0.0),
                "scale": np.array([1.0, 1.0, 1.0], np.float32)}

    def shared_full():
        return {"w1": _bf(rng, (I, D)),
                "w3": _bf(rng, (I, D)),
                "w2": _bf(rng, (D, I))}

    full = {}
    for l in range(cfg.n_layers):
        r = cfg.ratio(l)
        lay = {
            "attn_hc": hc(), "ffn_hc": hc(),
            "attn_norm": np.ones(D, np.float32),
            "ffn_norm": np.ones(D, np.float32),
            "wq_a": _bf(rng, (ql, D)), "q_norm": np.ones(ql, np.float32),
            "wq_b": _bf(rng, (Hl * dh, ql)),
            "wkv": _bf(rng, (dh, D)), "kv_norm": np.ones(dh, np.float32),
            "wo_a": _bf(rng, (ol, Hl * dh)),
            "wo_b": _bf(rng, (D, ol)),
            "attn_sink": _f32(rng, (Hl,), 0.1),
            "comp": None, "idx": None, "icomp": None,
            "gate": None, "shared": None,
        }
        if r:
            Dc = dh
            coff = 2 if r == 4 else 1
            lay["comp"] = {
                "wkv": _bf(rng, (coff * Dc, D)),
                "wgate": _bf(rng, (coff * Dc, D)),
                "ape": _f32(rng, (r, coff * Dc), 0.1),
                "norm": np.ones(Dc, np.float32),
            }
        if r == 4:
            ih = cfg.index_head_dim
            iHl = cfg.index_n_heads // d
            lay["idx"] = {
                "wq_b": _bf(rng, (iHl * ih, ql)),
                "weights_proj": _bf(rng, (iHl, D)),
            }
            lay["icomp"] = {
                "wkv": _bf(rng, (2 * ih, D)),
                "wgate": _bf(rng, (2 * ih, D)),
                "ape": _f32(rng, (r, 2 * ih), 0.1),
                "norm": np.ones(ih, np.float32),
            }
        gate = {"w": _bf(rng, (E, D))}
        if cfg.is_hash(l):
            # hash routing: tid2eid [V, K] int32 in [0, E)
            gate["tid2eid"] = rng.integers(0, E, (cfg.vocab_size,
                                                  cfg.top_k)).astype(np.int32)
        else:
            gate["bias"] = _f32(rng, (E,), 0.0)
        lay["gate"] = gate
        lay["shared"] = shared_full()
        full[l] = lay

    # ---- MTP layer ----
    mtp = {
        "attn_hc": hc(), "ffn_hc": hc(),
        "attn_norm": np.ones(D, np.float32),
        "ffn_norm": np.ones(D, np.float32),
        "wq_a": _bf(rng, (ql, D)), "q_norm": np.ones(ql, np.float32),
        "wq_b": _bf(rng, (Hl * dh, ql)),
        "wkv": _bf(rng, (dh, D)), "kv_norm": np.ones(dh, np.float32),
        "wo_a": _bf(rng, (ol, Hl * dh)),
        "wo_b": _bf(rng, (D, ol)),
        "attn_sink": _f32(rng, (Hl,), 0.1),
        "e_proj": _bf(rng, (D, D)), "h_proj": _bf(rng, (D, D)),
        "enorm": np.ones(D, np.float32), "hnorm": np.ones(D, np.float32),
        "norm": np.ones(D, np.float32),
        "gate": {"w": _bf(rng, (E, D)), "bias": _f32(rng, (E,), 0.0)},
        "shared": shared_full(),
        "hc_head_fn": _f32(rng, (Hc, Hc * D)),
        "hc_head_base": _f32(rng, (Hc,), 0.0),
        "hc_head_scale": np.array([1.0], np.float32),
    }

    head = {"fn": _f32(rng, (Hc, Hc * D)),
            "base": _f32(rng, (Hc,), 0.0),
            "scale": np.array([1.0], np.float32)}
    final_ln = np.ones(D, np.float32)

    # ---- shard across chips ----
    params_by_chip = []
    for c in range(d):
        chip = {}
        for l in range(cfg.n_layers):
            f = full[l]
            # shared expert: w1/w3 row-shard, w2 col-shard
            sh = {}
            for t in ("w1", "w3"):
                chunk = -(-f["shared"][t].shape[0] // d)
                sh[t] = f["shared"][t][c * chunk:(c + 1) * chunk]
            chunkc = -(-f["shared"]["w2"].shape[1] // d)
            sh["w2"] = f["shared"]["w2"][:, c * chunkc:(c + 1) * chunkc]
            lay = dict(f)  # shallow: replicated tensors shared OK
            lay["shared"] = sh
            chip[l] = lay
        msh = {}
        for t in ("w1", "w3"):
            chunk = -(-mtp["shared"][t].shape[0] // d)
            msh[t] = mtp["shared"][t][c * chunk:(c + 1) * chunk]
        chunkc = -(-mtp["shared"]["w2"].shape[1] // d)
        msh["w2"] = mtp["shared"]["w2"][:, c * chunkc:(c + 1) * chunkc]
        m2 = dict(mtp)
        m2["shared"] = msh
        chip["mtp"] = m2
        chip["head"] = head
        chip["final_ln"] = final_ln
        params_by_chip.append(chip)

    # ---- host experts (fp4) ----
    embed = _bf(rng, (cfg.vocab_size, D))
    lm_head = _bf(rng, (cfg.vocab_size, D))
    expert_host = {}
    keys = list(range(cfg.n_layers)) + ["mtp"]
    for key in keys:
        for e in range(E):
            ex = {}
            for t, out, in_ in (("w1", I, D), ("w3", I, D), ("w2", D, I)):
                w = (rng.standard_normal((out, in_)) * 0.1).astype(np.float32)
                p, s = quantize_np(w)
                ex[t] = p
                ex[t + "_s"] = s
            expert_host[(key, e)] = ex
    return params_by_chip, embed, lm_head, expert_host
'''
    MODS['dsv4_runtime.py'] = '''"""pmap runtime for DeepSeek-V4-Flash TPU serving (de-simplified).

Sites (per layer type; static shapes; prefill-S and decode-S=1 compile
separately):
  site_attn(p, streams, valid, pos0, <layer state>)   sliding/CSA/HCA
  site_ffn(p, bank, streams, input_ids [, collect])   MoE (every layer)
  site_mtp(p, bank, streams_h, embed, id, pos0, ring) MTP draft layer
  _final(streams, ...)                                hc_head collapse

Per-layer KV state (replicated — MQA; every chip needs all KV):
  ring:   u8 [B,W,448] + scales f32 [B,W,7] + rope bf16 [B,W,64]
  comp:   u8 [B,C,448] + scales [B,C,7] + rope [B,C,64]  (C=max_ctx//r)
  idxr:   bf16 [B,C,128] keys-only (CSA layers)
  cstate: compressor carry (kv, score) [B, coff*ratio, coff*Dc] f32
  icstate: indexer compressor carry (CSA layers)
Positions tracked host-side (cache_len); passed as traced pos0.

Prefill: chunks of cfg.prefill_chunk (multiple of every ratio), MoE =
affine-correction full sweep (disjoint banks; exact for any routing).
Decode: 1 token; hot banks + exact two-phase refresh fixpoint
(snapshot -> run -> miss? refresh + rollback + re-run).
MTP-1: draft(streams, next_token) -> logits; verify loop in generate().
"""
from __future__ import annotations

import time

import numpy as np

import jax
import jax.numpy as jnp
from jax import lax, pmap

from . import dsv4_layers as dl
from .dsv4_config import Dsv4Config
from .dsv4_fp4 import dequant_jax, fp4_sim_jax


def _dg(a):
    return np.asarray(jax.device_get(a))


# ===========================================================================
# attention core (one chip, local heads)
# ===========================================================================

def _attn_core(p, streams, valid, pos0, ring, cstate, comp, idxr, icstate,
               cfg, d, mf, cf, ratio):
    """One attention site on one chip (local heads = n_heads // d).
    ring = (u8, s, rope); cstate = (kv_state, sc_state);
    comp = (u8, s, rope); idxr [B,C,ih]; icstate = (kv, sc).
    ratio: STATIC int.  Returns (streams', new_state tuple)."""
    post, comb, collapsed = dl.hc_site(p["attn_hc"], streams, cfg)
    h = dl.rms_norm(collapsed, p["attn_norm"], cfg.rms_norm_eps)
    B, S = h.shape[0], h.shape[1]
    Hl = cfg.n_heads // d
    dh = cfg.head_dim
    W = cfg.window_size
    freqs = cf if ratio else mf
    pos = pos0 + jnp.arange(S)

    # ---- q: low-rank + per-head RMSNorm + rope (trailing rd dims) ----
    qr = dl.rms_norm(h @ p["wq_a"].astype(jnp.bfloat16).T, p["q_norm"],
                     cfg.rms_norm_eps)
    q = (qr @ p["wq_b"].astype(jnp.bfloat16).T).reshape(B, S, Hl, dh)
    q32 = q.astype(jnp.float32)
    q32 = q32 * lax.rsqrt(jnp.mean(q32 * q32, -1, keepdims=True)
                          + cfg.rms_norm_eps)
    q = dl.apply_rope(q32.astype(jnp.bfloat16), pos, freqs)

    # ---- shared-KV MQA ----
    kv = dl.rms_norm(h @ p["wkv"].astype(jnp.bfloat16).T, p["kv_norm"],
                     cfg.rms_norm_eps)
    kv = dl.apply_rope(kv, pos, freqs)

    # ---- ring write (chunk's last min(S, W) tokens) ----
    u8, sc, rp = dl.kv_pack(kv, cfg.rope_head_dim)
    n_write = min(S, W)
    u8w, scw = u8[:, -n_write:], sc[:, -n_write:]
    rpw = rp[:, -n_write:]
    slots = (pos0 + S - n_write + jnp.arange(n_write)) % W
    r_u8 = ring[0].at[:, slots].set(u8w)
    r_s = ring[1].at[:, slots].set(scw)
    r_r = ring[2].at[:, slots].set(rpw.astype(jnp.float32))

    # ---- ring entry positions ----
    last = pos0 - 1
    g = last - ((last - jnp.arange(W)) % W)
    g = jnp.where(g >= 0, g, -1)

    ring_kv = dl.kv_unpack(ring[0], ring[1], ring[2].astype(jnp.bfloat16))
    ctx_kv = jnp.concatenate([ring_kv, kv], axis=1)
    ctx_pos = jnp.concatenate([g, pos])
    W_ = ctx_pos.shape[0]

    c_u8, c_s, c_r = comp[0], comp[1], comp[2]
    idx_new = idxr
    in_state = icstate
    cnk, cns = cstate
    comp_kv_sel = None

    if ratio:
        # ------------- compressor (CSA ratio 4 / HCA 8|128) -------------
        if S == 1:
            ent, should, (nk, ns) = dl.compressor_decode(
                p["comp"], h, cstate, pos0, cfg)
            n_new = 1
        else:
            ent, (nk, ns) = dl.compressor_prefill(
                p["comp"], h, cstate, pos0, cfg)
            n_new = ent.shape[1]
        # compressed-entry positions: e*ratio for the n_new new entries
        cpos = (pos0 // ratio + jnp.arange(n_new)) * ratio
        ent = dl.apply_rope(ent, cpos, cf)
        e0 = pos0 // ratio
        cu8, csc, crp = dl.kv_pack(ent, cfg.rope_head_dim)
        c_u8 = lax.dynamic_update_slice(comp[0], cu8, (0, e0, 0))
        c_s = lax.dynamic_update_slice(comp[1], csc, (0, e0, 0))
        c_r = lax.dynamic_update_slice(
            comp[2], crp.astype(jnp.float32), (0, e0, 0))
        comp_kv = dl.kv_unpack(c_u8, c_s, c_r)          # [B,C,dh]
        comp_idx = jnp.arange(comp_kv.shape[1])
        cnk, cns = nk, ns
        Ctot = comp_kv.shape[1]

        if ratio == 4:
            # ---------------- indexer (CSA) ----------------
            if S == 1:
                ient, ish, (ink, ins) = dl.compressor_decode(
                    p["icomp"], h, icstate, pos0, cfg)
                n_in = 1
            else:
                ient, (ink, ins) = dl.compressor_prefill(
                    p["icomp"], h, icstate, pos0, cfg)
                n_in = ient.shape[1]
            ipos = (pos0 // ratio + jnp.arange(n_in)) * ratio
            ient = dl.apply_rope(ient, ipos, cf)
            ih = dl.hadamard(ient)
            ihq = fp4_sim_jax(ih.astype(jnp.float32), 32)
            i0 = pos0 // ratio
            idx_new = lax.dynamic_update_slice(
                idxr, ihq.astype(jnp.float32), (0, i0, 0))
            iscores = _indexer_scores(
                p["idx"], qr, h, idx_new, pos0, cfg, d, cf)
            thr = (pos[:, None] + 1) // ratio
            valid_e = comp_idx[None, :] < thr
            iscores = jnp.where(valid_e, iscores, -1e30)
            k = min(cfg.index_topk, Ctot)
            tvals, sel = lax.top_k(iscores, k)          # [B,S,k]
            sel = jnp.where(tvals > -1e29, sel, -1)
            safe = jnp.where(sel >= 0, sel, 0)
            # gath[b,s,j,:] = comp_kv[b, safe[b,s,j], :]
            bb = jnp.broadcast_to(jnp.arange(B)[:, None, None],
                                  (B, S, k))
            gath = comp_kv[bb, safe]                     # [B,S,k,dh]
            gath = jnp.where((sel >= 0)[:, :, :, None], gath, 0.0)
            gath_pos = comp_idx[safe]                  # [B,S,k]
            gath_pos = jnp.where(sel >= 0, gath_pos, -1)
            comp_kv_sel = gath.reshape(B, S * k, dh)
            comp_pos_sel = gath_pos.reshape(B, S * k)
            in_state = (ink, ins)
        else:
            comp_kv_sel = comp_kv
            comp_pos_sel = jnp.broadcast_to(comp_idx[None, :],
                                            (B, Ctot))

        kv_sel = jnp.concatenate([ctx_kv, comp_kv_sel], axis=1)
        pos_sel = jnp.concatenate(
            [ctx_pos,
             comp_pos_sel[0] if comp_pos_sel.ndim == 2 else comp_pos_sel],
            axis=0)
        is_comp = jnp.concatenate([jnp.zeros(W_, jnp.bool_),
                                   jnp.ones(kv_sel.shape[1] - W_,
                                            jnp.bool_)])
    else:
        kv_sel, pos_sel = ctx_kv, ctx_pos
        is_comp = jnp.zeros(kv_sel.shape[1], jnp.bool_)

    N = kv_sel.shape[1]
    pq = pos[:, None]                                  # [S,1]
    pk = pos_sel[None, :]                              # [1,N]
    causal = pk <= pq                                  # [S,N]
    in_win = (pk > pq - W) & (pk >= 0)
    vis = jnp.where(is_comp[None, :], causal, causal & in_win)
    valid_ctx = jnp.broadcast_to(vis[None], (B, S, N))

    o = dl.sparse_attn_core(q, kv_sel, valid_ctx.astype(jnp.float32),
                            p["attn_sink"], dh ** -0.5)
    o = dl.apply_rope(o, pos, freqs, inverse=True)
    out = dl.grouped_o_proj(o, p["wo_a"], p["wo_b"])
    out = lax.psum(out, "tp")
    streams2 = dl.hc_apply(post, comb, out.astype(streams.dtype), streams)
    streams2 = streams2 * valid[..., None, None].astype(streams2.dtype)

    new_state = (r_u8, r_s, r_r, c_u8, c_s, c_r, idx_new,
                 in_state[0], in_state[1], cnk, cns)
    return streams2, new_state


def _indexer_scores(p_idx, qr, x, idx_cache, pos0, cfg, d, cf):
    """Head-sharded indexer scores (psum over chips).  qr [B,S,ql];
    idx_cache [B,Ci,ih] bf16 -> scores [B,S,Ci] f32."""
    q = qr @ p_idx["wq_b"].astype(jnp.bfloat16).T
    Hl = p_idx["wq_b"].shape[0] // cfg.index_head_dim
    q = q.reshape(qr.shape[0], qr.shape[1], Hl, cfg.index_head_dim)
    pos = pos0 + jnp.arange(qr.shape[1])
    q = dl.apply_rope(q, pos, cf)
    q = dl.hadamard(q)
    q = fp4_sim_jax(q.astype(jnp.float32), 32).astype(jnp.bfloat16)
    w = (x @ p_idx["weights_proj"].astype(jnp.bfloat16).T) \
        * (cfg.index_head_dim ** -0.5 * (Hl * d) ** -0.5)
    s = jnp.einsum("bshd,btd->bsht", q.astype(jnp.float32),
                   idx_cache.astype(jnp.float32))
    s = jnp.maximum(s, 0.0)
    s = jnp.einsum("bsht,bsh->bst", s, w.astype(jnp.float32))
    return lax.psum(s, "tp")


_INDEX_FREQS = None   # set by runner at compile time (module-level hack
                     # avoided: passed via p dict below instead)



def _hc_head_final(streams, hfn, hbase, hscale, w, cfg):
    """Model head: hc_head collapse (sigmoid pre-weights, no Sinkhorn) +
    final RMSNorm.  streams [..., Hc, D] (any leading axes)."""
    lead = streams.shape[:-2]
    H, D = streams.shape[-2:]
    flat = streams.reshape(*lead, H * D).astype(jnp.float32)
    flat = dl.rms_norm_no_w(flat, cfg.rms_norm_eps)
    mixes = flat @ hfn.astype(jnp.float32).T
    pre = jax.nn.sigmoid(mixes * hscale[0] + hbase) + cfg.hc_eps
    y = jnp.sum(pre[..., None] * streams, axis=2)
    return dl.rms_norm(y.astype(streams.dtype), w, cfg.rms_norm_eps)


class Dsv4Runner:
    # ================================================================ setup
    def __init__(self, cfg: Dsv4Config, params_by_chip, embed_np, lm_head_np,
                 expert_host, log=print):
        global _INDEX_FREQS
        self.cfg = cfg
        self.log = log
        self.devs = jax.devices()
        self.d = len(self.devs)
        self.expert_host = expert_host
        self.embed_np = embed_np.astype(np.float32)
        self.lm_head_np = lm_head_np.astype(np.float32)
        self.PJ = {}
        for l in range(cfg.n_layers):
            self.PJ[l] = self._stack(
                [params_by_chip[c][l] for c in range(self.d)])
        self.PJ["mtp"] = self._stack(
            [params_by_chip[c]["mtp"] for c in range(self.d)])
        self.head_params = params_by_chip[0]["head"]
        self.final_norm = jnp.asarray(params_by_chip[0]["final_ln"])
        self.sharding_tp = jax.sharding.NamedSharding(
            jax.sharding.Mesh(np.array(self.devs), ("tp",)),
            jax.sharding.PartitionSpec("tp"))
        self.sharding_rep = jax.sharding.NamedSharding(
            jax.sharding.Mesh(np.array(self.devs), ("tp",)),
            jax.sharding.PartitionSpec())
        self.n_slots = cfg.n_slots
        self.banks = {}
        self.bank_ids = {}
        self._last_routed = {}
        self._build_freqs()
        _INDEX_FREQS = self.cf
        self.state = self._init_state()
        self._init_banks()
        self._build_sites()

    def _build_freqs(self):
        cfg = self.cfg
        self.mf = jnp.asarray(dl.yarn_freqs(cfg, False))
        self.cf = jnp.asarray(dl.yarn_freqs(cfg, True))

    def _mark(self, chip_lay, l):
        out = dict(chip_lay)
        out["_l"] = l if isinstance(l, int) else -1   # mtp = -1
        return out

    def _stack(self, chip_dicts):
        """Stack leaves across chips on a new leading axis.  None leaves
        (unused compressor/indexer on sliding layers) become dummy [1]
        arrays; the int layer marker is kept un-stacked."""
        def st(*xs):
            if xs[0] is None:
                return jnp.zeros((len(xs), 1), jnp.float32)
            return jnp.stack([jnp.asarray(x) for x in xs], axis=0)
        tree = jax.tree.map(st, *chip_dicts,
                            is_leaf=lambda t: t is None)
        return tree

    # ------------------------------------------------------------- states
    def _init_state(self):
        cfg = self.cfg
        B, W = 1, cfg.window_size
        st = {"cache_len": 0}
        st["ring"] = []
        st["comp"] = []
        st["idxr"] = []
        st["cstate"] = []
        st["icstate"] = []
        Dc, rd = cfg.head_dim, cfg.rope_head_dim
        for l in range(cfg.n_layers):
            r = cfg.ratio(l)
            # every layer (sliding included) has the W-entry ring
            st["ring"].append((
                self._sh(np.zeros((B, W, Dc - rd), np.uint8)),
                self._sh(np.zeros((B, W, (Dc - rd) // 64), np.float32)),
                self._sh(np.zeros((B, W, rd), np.float32))))
            if r == 0:
                st["comp"].append(None)
                st["idxr"].append(None)
                st["cstate"].append(None)
                st["icstate"].append(None)
                continue
            C = cfg.n_comp(r)
            coff = 2 if r == 4 else 1
            st["comp"].append((
                self._sh(np.zeros((B, C, Dc - rd), np.uint8)),
                self._sh(np.zeros((B, C, (Dc - rd) // 64), np.float32)),
                self._sh(np.zeros((B, C, rd), np.float32))))
            st["cstate"].append((
                self._sh(np.zeros((B, coff * r, coff * Dc), np.float32)),
                self._sh(np.zeros((B, coff * r, coff * Dc), np.float32))))
            if r == 4:
                ic = 2 * cfg.index_head_dim
                st["idxr"].append(self._sh(np.zeros(
                    (B, C, cfg.index_head_dim), np.float32)))
                st["icstate"].append((
                    self._sh(np.zeros((B, 2 * r, ic), np.float32)),
                    self._sh(np.zeros((B, 2 * r, ic), np.float32))))
            else:
                st["idxr"].append(None)
                st["icstate"].append(None)
        st["mtp_ring"] = (
            self._sh(np.zeros(
                (B, W, cfg.head_dim - cfg.rope_head_dim), np.uint8)),
            self._sh(np.zeros(
                (B, W, (cfg.head_dim - cfg.rope_head_dim) // 64),
                np.float32)),
            self._sh(np.zeros((B, W, cfg.rope_head_dim), np.float32)))
        return st

    def _sh(self, arr):
        return jax.device_put(np.stack([arr] * self.d), self.sharding_tp)

    def _sc(self, v):
        return jax.device_put(
            np.full((self.d,), int(v), np.int32), self.sharding_tp)

    # ------------------------------------------------------------- banks
    def _bank_np(self, key, ids_per_chip):
        cfg = self.cfg
        I, D = cfg.moe_inter, cfg.hidden_size
        n = self.n_slots
        outs = []
        for dev_ids in ids_per_chip:
            bank = {
                "w1": np.zeros((n, I, D // 2), np.uint8),
                "w1_s": np.zeros((n, I, D // 32), np.uint8),
                "w3": np.zeros((n, I, D // 2), np.uint8),
                "w3_s": np.zeros((n, I, D // 32), np.uint8),
                "w2": np.zeros((n, D, I // 2), np.uint8),
                "w2_s": np.zeros((n, D, I // 32), np.uint8),
                "ids": np.asarray(dev_ids[:n], np.int32),
            }
            for s in range(min(n, len(dev_ids))):
                e = int(bank["ids"][s])
                if e < 0:
                    continue
                ex = self.expert_host[(key, e)]
                for t in ("w1", "w3", "w2"):
                    bank[t][s] = ex[t]
                    bank[t + "_s"][s] = ex[t + "_s"]
            outs.append(bank)
        return outs

    def _install_bank(self, key, ids_per_chip):
        per_chip = self._bank_np(key, ids_per_chip)
        self.banks[key] = jax.device_put(
            {k: np.stack([pc[k] for pc in per_chip], axis=0)
             for k in per_chip[0]}, self.sharding_tp)
        self.bank_ids[key] = [list(map(int, pc["ids"])) for pc in per_chip]

    def _init_banks(self):
        empty = [[-1] * self.n_slots] * self.d
        for l in range(self.cfg.n_layers):
            self._install_bank(l, empty)
        self._install_bank("mtp", empty)

    # ------------------------------------------------------------- compile
    def _build_sites(self):
        cfg, d = self.cfg, self.d
        mf, cf = self.mf, self.cf

        def attn_fn(p, streams, valid, pos0, ring_u8, ring_s, ring_r,
                    c_kv, c_sc, i_kv, i_sc, comp_u8, comp_s, comp_r, idxr,
                    ratio):
            return _attn_core(p, streams, valid, pos0,
                              (ring_u8, ring_s, ring_r), (c_kv, c_sc),
                              (comp_u8, comp_s, comp_r), idxr,
                              (i_kv, i_sc), cfg, d, mf, cf, ratio)

        def _ffn(p, bank, streams, input_ids, collect=False):
            post, comb, collapsed = dl.hc_site(p["ffn_hc"], streams, cfg)
            h = dl.rms_norm(collapsed, p["ffn_norm"], cfg.rms_norm_eps)
            w, ids = dl.moe_router(p["gate"], h, input_ids, cfg)
            sh = dl.dense_mlp_core(p["shared"], h, cfg)
            moe = dl.fp4_bank_core(bank, h, w, ids, cfg)
            y = sh + moe
            streams2 = dl.hc_apply(post, comb, y.astype(streams.dtype),
                                   streams)
            if collect:
                return streams2, ids
            return streams2

        def mtp_fn(p, bank, streams_h, embed_tok, input_id, pos0,
                   r_u8, r_s, r_r):
            e = dl.rms_norm(embed_tok[None], p["enorm"], cfg.rms_norm_eps)
            hprev = dl.rms_norm(streams_h[:, 0], p["hnorm"],
                                cfg.rms_norm_eps)
            x = (e @ p["e_proj"].astype(jnp.bfloat16).T
                 + hprev @ p["h_proj"].astype(jnp.bfloat16).T)
            streams = jnp.broadcast_to(
                x[:, :, None, :],
                (x.shape[0], x.shape[1], cfg.hc_mult,
                 cfg.hidden_size)).astype(streams_h.dtype)
            post, comb, collapsed = dl.hc_site(p["attn_hc"], streams, cfg)
            hn = dl.rms_norm(collapsed, p["attn_norm"], cfg.rms_norm_eps)
            B, S = hn.shape[0], hn.shape[1]
            Hl = cfg.n_heads // d
            dh = cfg.head_dim
            W = cfg.window_size
            pos = pos0 + jnp.arange(S)
            qr = dl.rms_norm(hn @ p["wq_a"].astype(jnp.bfloat16).T,
                             p["q_norm"], cfg.rms_norm_eps)
            q = (qr @ p["wq_b"].astype(jnp.bfloat16).T).reshape(B, S, Hl, dh)
            q32 = q.astype(jnp.float32)
            q32 = q32 * lax.rsqrt(jnp.mean(q32 * q32, -1, keepdims=True)
                                  + cfg.rms_norm_eps)
            q = dl.apply_rope(q32.astype(jnp.bfloat16), pos, mf)
            kv = dl.rms_norm(hn @ p["wkv"].astype(jnp.bfloat16).T,
                             p["kv_norm"], cfg.rms_norm_eps)
            kv = dl.apply_rope(kv, pos, mf)
            u8, sc, rp = dl.kv_pack(kv, cfg.rope_head_dim)
            slots = pos % W
            n_u8 = r_u8.at[:, slots[0]].set(u8[:, 0])
            n_s = r_s.at[:, slots[0]].set(sc[:, 0])
            n_r = r_r.at[:, slots[0]].set(rp[:, 0].astype(jnp.float32))
            ring_kv = dl.kv_unpack(n_u8, n_s, n_r.astype(jnp.bfloat16))
            last = pos0 - 1
            g = last - ((last - jnp.arange(W)) % W)
            g = jnp.where(g >= 0, g, -1)
            vis = (g[None, :] <= pos[:, None]) \
                & (g[None, :] > pos[:, None] - W) & (g[None, :] >= 0)
            valid_ctx = jnp.broadcast_to(vis[None], (B, S, W))
            o = dl.sparse_attn_core(q, ring_kv,
                                    valid_ctx.astype(jnp.float32),
                                    p["attn_sink"], dh ** -0.5)
            o = dl.apply_rope(o, pos, mf, inverse=True)
            out = dl.grouped_o_proj(o, p["wo_a"], p["wo_b"])
            out = lax.psum(out, "tp")
            streams2 = dl.hc_apply(post, comb, out.astype(streams.dtype),
                                   streams)
            post2, comb2, collapsed2 = dl.hc_site(p["ffn_hc"], streams2, cfg)
            hn2 = dl.rms_norm(collapsed2, p["ffn_norm"], cfg.rms_norm_eps)
            w, ids = dl.moe_router(p["gate"], hn2, input_id[:, None], cfg)
            sh = dl.dense_mlp_core(p["shared"], hn2, cfg)
            moe = dl.fp4_bank_core(bank, hn2, w, ids, cfg)
            y = sh + moe
            streams3 = dl.hc_apply(post2, comb2, y.astype(streams2.dtype),
                                   streams2)
            hcol = _hc_head_final(streams3, p["hc_head_fn"],
                                  p["hc_head_base"], p["hc_head_scale"],
                                  p["norm"], cfg)
            return hcol, ids, (n_u8, n_s, n_r)

        self.site_attn = pmap(attn_fn, axis_name="tp",
                              static_broadcasted_argnums=(15,))
        self._ffn_plain = pmap(
            lambda p, bank, streams, ids: _ffn(p, bank, streams, ids),
            axis_name="tp")
        self._ffn_collect = pmap(
            lambda p, bank, streams, ids: _ffn(p, bank, streams, ids,
                                               True), axis_name="tp")
        self.site_mtp = pmap(mtp_fn, axis_name="tp")
        self._final = jax.jit(
            lambda streams: _hc_head_final(
                streams, self.head_params["fn"],
                self.head_params["base"], self.head_params["scale"],
                self.final_norm, cfg))

    # ------------------------------------------------------------- helpers
    def _attn_call(self, l, streams, valid, pos0):
        cfg = self.cfg
        p = self.PJ[l]
        st = self.state
        r = cfg.ratio(l)
        zc = self._zero_cstate()
        zi = self._zero_icstate()
        zcomp = self._zero_comp()
        zidx = self._zero_idxr()
        ring = st["ring"][l]
        comp = st["comp"][l] if r else zcomp
        cs = st["cstate"][l] if r else zc
        ics = st["icstate"][l] if r == 4 else zi
        idxr = st["idxr"][l] if r == 4 else zidx
        out = self.site_attn(p, streams, valid, self._sc(pos0),
                             ring[0], ring[1], ring[2],
                             cs[0], cs[1], ics[0], ics[1],
                             comp[0], comp[1], comp[2], idxr, r)
        streams2, ns = out
        st["ring"][l] = (ns[0], ns[1], ns[2])
        if r:
            st["comp"][l] = (ns[3], ns[4], ns[5])
            st["cstate"][l] = (ns[9], ns[10])
            if r == 4:
                st["idxr"][l] = ns[6]
                st["icstate"][l] = (ns[7], ns[8])
        return streams2

    def _zero_cstate(self):
        if not hasattr(self, "_zc"):
            cfg = self.cfg
            self._zc = (self._sh(np.zeros((1, 8, 2 * cfg.head_dim),
                                          np.float32)),) * 2
        return self._zc

    def _zero_icstate(self):
        if not hasattr(self, "_zic"):
            cfg = self.cfg
            self._zic = (self._sh(np.zeros(
                (1, 8, 2 * cfg.index_head_dim), np.float32)),) * 2
        return self._zic

    def _zero_comp(self):
        if not hasattr(self, "_zcomp"):
            cfg = self.cfg
            Dc, rd = cfg.head_dim, cfg.rope_head_dim
            self._zcomp = (
                self._sh(np.zeros((1, 1, Dc - rd), np.uint8)),
                self._sh(np.zeros((1, 1, (Dc - rd) // 64), np.float32)),
                self._sh(np.zeros((1, 1, rd), np.float32)))
        return self._zcomp

    def _zero_idxr(self):
        if not hasattr(self, "_zidx"):
            self._zidx = self._sh(np.zeros(
                (1, 1, self.cfg.index_head_dim), np.float32))
        return self._zidx

    def _ffn_call(self, l, streams, input_ids, collect=False):
        p = self.PJ[l]
        if collect:
            return self._ffn_collect(p, self.banks[l], streams, input_ids)
        return self._ffn_plain(p, self.banks[l], streams, input_ids)

    # ------------------------------------------------------------- prefill
    def reset(self):
        self.state = self._init_state()

    def _embed_streams(self, tokens, valid):
        cfg = self.cfg
        x = self.embed_np[np.asarray(tokens, np.int32)] \
            * np.asarray(valid, np.float32)[:, None]
        B, S = 1, len(tokens)
        streams = np.broadcast_to(
            x[None, :, None, :], (B, S, cfg.hc_mult, cfg.hidden_size))
        streams = np.ascontiguousarray(streams, dtype=np.float32)
        return jax.device_put(np.stack([streams] * self.d),
                              self.sharding_tp)

    def prefill(self, tokens):
        cfg = self.cfg
        self.reset()
        S = cfg.prefill_chunk
        n = len(tokens)
        pad = (-n) % S
        toks = [0] * pad + list(tokens)
        pos0 = 0
        last_hidden = None
        for ci in range(0, len(toks), S):
            chunk = toks[ci:ci + S]
            n_real = min(S, n - (ci - pad))
            valid_l = [0.0] * (S - n_real) + [1.0] * n_real
            streams = self._embed_streams(chunk, valid_l)
            valid = self._sh(np.asarray(valid_l, np.float32).reshape(1, S))
            ids = self._sh(np.asarray(chunk, np.int32).reshape(1, S))
            for l in range(cfg.n_layers):
                streams = self._attn_call(l, streams, valid, pos0)
                streams, routed = self._prefill_moe(l, streams, ids)
                self._last_routed[l] = _dg(routed[0])
            h = self._final(streams)
            last_hidden = _dg(h[0, 0, -1])
            self.state["cache_len"] += n_real
            pos0 += n_real
        self._last_hidden = last_hidden
        self._last_streams = streams
        self._refresh_all_banks_for_decode()
        return last_hidden

    def _prefill_moe(self, l, streams, ids):
        """Exact MoE during prefill via the affine-correction sweep
        (mHC site output is affine in the sublayer output; sweeping
        disjoint banks and subtracting (P-1) empty-bank sites is
        exact)."""
        cfg = self.cfg
        E = cfg.n_experts
        step = self.d * self.n_slots
        n_passes = -(-E // step)
        empty = [[-1] * self.n_slots] * self.d
        self._install_bank(l, empty)
        base, routed = self._ffn_call(l, streams, ids, collect=True)
        total = None
        for p_i in range(n_passes):
            start = p_i * step
            per_chip = []
            for dev in range(self.d):
                lo = start + dev * self.n_slots
                hi = min(lo + self.n_slots, E)
                per_chip.append(list(range(lo, hi))
                                + [-1] * max(0, self.n_slots - (hi - lo)))
            self._install_bank(l, per_chip)
            streams_p, routed = self._ffn_call(l, streams, ids,
                                               collect=True)
            total = streams_p if total is None else total + streams_p
        result = total - (n_passes - 1) * base
        return result, routed

    def _refresh_all_banks_for_decode(self):
        for l in range(self.cfg.n_layers):
            ids_l = self._last_routed.get(l)
            if ids_l is None:
                continue
            arr = np.asarray(ids_l).reshape(-1, self.cfg.top_k)
            last = [int(i) for i in arr[-1]]
            freq = {}
            for i in arr.flatten():
                freq[int(i)] = freq.get(int(i), 0) + 1
            priority = list(dict.fromkeys(last))
            for i, _ in sorted(freq.items(), key=lambda kv: -kv[1]):
                if i not in priority:
                    priority.append(i)
            cap = self.d * self.n_slots
            chosen = priority[:cap]
            per_chip = [chosen[c::self.d][:self.n_slots]
                        for c in range(self.d)]
            per_chip = [pc + [-1] * (self.n_slots - len(pc))
                        for pc in per_chip]
            self._install_bank(l, per_chip)

    # ------------------------------------------------------------- decode
    def _decode_one(self, token, temperature=0.0, top_p=1.0):
        cfg = self.cfg
        pos = self.state["cache_len"]
        streams = self._embed_streams([token], [1.0])
        valid = self._sh(np.ones((1, 1), np.float32))
        ids = self._sh(np.asarray([[token]], np.int32))
        max_iters = cfg.n_layers + 2
        for _ in range(max_iters):
            snap = self._snapshot_state()
            streams_run, routed, hidden = self._run_all(streams, valid, ids,
                                                        pos, collect=True)
            missing = self._missing(routed)
            if not missing:
                logits = self._lm_head(hidden)
                self._last_streams = streams_run
                return logits, streams_run
            self._refresh(missing)
            self._restore_state(snap)
        raise RuntimeError("decode bank fixpoint did not converge")

    def _run_all(self, streams, valid, ids, pos, collect):
        cfg = self.cfg
        routed = {}
        for l in range(cfg.n_layers):
            streams = self._attn_call(l, streams, valid, pos)
            if collect:
                streams, r = self._ffn_call(l, streams, ids, collect=True)
                routed[l] = _dg(r[0])
            else:
                streams = self._ffn_call(l, streams, ids)
        self.state["cache_len"] = pos + 1
        h = self._final(streams)
        return streams, routed, _dg(h[0, 0, 0])

    def _missing(self, routed):
        miss = {}
        for l, ids in routed.items():
            have = set()
            for pc in self.bank_ids[l]:
                have.update(e for e in pc if e >= 0)
            need = set(int(i) for i in np.asarray(ids).flatten())
            if not need <= have:
                miss[l] = need
        return miss

    def _refresh(self, missing):
        for l, need in missing.items():
            have = [e for pc in self.bank_ids[l] for e in pc if e >= 0]
            old = list(dict.fromkeys(have))
            new_ids = list(dict.fromkeys(list(need) + old))[
                :self.d * self.n_slots]
            per_chip = [new_ids[c::self.d] for c in range(self.d)]
            per_chip = [pc + [-1] * (self.n_slots - len(pc))
                        for pc in per_chip]
            self._install_bank(l, per_chip)

    def _snapshot_state(self):
        st = self.state
        return {k: (list(v) if isinstance(v, list) else v)
                for k, v in st.items()}

    def _restore_state(self, snap):
        self.state = snap

    # ------------------------------------------------------------- MTP
    def draft(self, streams, next_token):
        """MTP-1 draft: target hc-streams + next token -> logits for the
        position after it.  Draft attention position = cache_len."""
        cfg = self.cfg
        p = self.PJ["mtp"]
        pos = self.state["cache_len"]
        embed = jax.device_put(
            np.stack([self.embed_np[next_token]] * self.d),
            self.sharding_tp)
        ids = self._sh(np.asarray([[next_token]], np.int32))
        pos0 = self._sc(pos)
        max_iters = 3
        for _ in range(max_iters):
            ring = self.state["mtp_ring"]
            out = self.site_mtp(p, self.banks["mtp"], streams, embed,
                                ids, pos0, ring[0], ring[1], ring[2])
            hcol, routed, new_ring = out
            need = set(int(i) for i in np.asarray(_dg(routed[0])).flatten())
            have = set()
            for pc in self.bank_ids["mtp"]:
                have.update(e for e in pc if e >= 0)
            if need <= have:
                self.state["mtp_ring"] = new_ring
                return self._lm_head(_dg(hcol[0, 0, 0]))
            old_ids = list(dict.fromkeys(
                [e for pc in self.bank_ids["mtp"] for e in pc if e >= 0]))
            new_ids = list(dict.fromkeys(list(need) + old_ids))[
                :self.d * self.n_slots]
            per_chip = [new_ids[c::self.d] for c in range(self.d)]
            per_chip = [pc + [-1] * (self.n_slots - len(pc))
                        for pc in per_chip]
            self._install_bank("mtp", per_chip)
        raise RuntimeError("mtp draft fixpoint did not converge")

    # ------------------------------------------------------------- logits
    def _lm_head(self, hidden):
        h = np.asarray(hidden).reshape(-1)
        return h @ self.lm_head_np.T

    # ------------------------------------------------------------- gen
    def generate(self, tokens, max_new_tokens=64, temperature=0.0,
                 top_p=1.0, stop_ids=None, on_token=None):
        h = self.prefill(tokens)
        logits = self._lm_head(h)
        out = []
        for i in range(max_new_tokens):
            t = self._sample(logits, temperature, top_p)
            if stop_ids and t in stop_ids:
                break
            out.append(t)
            if on_token:
                on_token(t)
            if i + 1 < max_new_tokens:
                logits, _ = self._decode_one(t, temperature, top_p)
        return out

    def generate_mtp(self, tokens, max_new_tokens=64, temperature=0.0,
                     top_p=1.0, stop_ids=None, on_token=None, stats=None):
        """MTP-1 speculative decode.  Greedy path is lossless: emits
        a=argmax(target L) always, plus d when the draft's d equals the
        target's next argmax (verified by the target step on a)."""
        h = self.prefill(tokens)
        logits = self._lm_head(h)
        streams = self._last_streams
        out = []
        n_acc = n_rej = 0
        while len(out) < max_new_tokens:
            a = self._sample(logits, temperature, top_p)
            if stop_ids and a in stop_ids:
                break
            out.append(a)
            if on_token:
                on_token(a)
            draft_logits = self.draft(streams, a)
            d = int(np.argmax(draft_logits))
            logits, streams = self._decode_one(a, temperature, top_p)
            emitted = a
            if int(np.argmax(logits)) == d:
                # verified: d is exactly the target's next token
                if not (stop_ids and d in stop_ids) \
                        and len(out) < max_new_tokens:
                    out.append(d)
                    emitted = d
                    if on_token:
                        on_token(d)
                n_acc += 1
                # next cycle consumes d (target runs on d next round);
                # but we need logits for the position after d:
                logits, streams = self._decode_one(emitted, temperature,
                                                   top_p)
            else:
                n_rej += 1
        if stats is not None:
            stats.update({"accepts": n_acc, "rejects": n_rej})
        return out

    def _sample(self, logits, temperature, top_p):
        v = np.asarray(logits).reshape(-1)
        if temperature <= 1e-6:
            return int(np.argmax(v))
        v = v / temperature
        v = v - v.max()
        p = np.exp(v)
        p = p / p.sum()
        if top_p and top_p < 1.0:
            order = np.argsort(-p)
            cum = np.cumsum(p[order])
            cut = np.searchsorted(cum, top_p) + 1
            keep = order[:cut]
            p2 = p[keep] / p[keep].sum()
            return int(np.random.default_rng().choice(keep, p=p2))
        return int(np.random.default_rng().choice(len(p), p=p))
'''
    MODS['dsv4_loader.py'] = '''"""Real-weight loader for DeepSeek-V4-Flash (env-driven repo switch).

Streams safetensors shards from HuggingFace into host RAM and carves
them into the engine's parameter structure (mirrors dsv4_params layout).

Repo selection (like loader_real.py):
  DSV4_REPO env (default deepseek-ai/DeepSeek-V4-Flash, ungated MIT).
  The notebook sets DSV4_REPO=orcarouter/DeepSeek-V4-Flash-Vision-
  Uncensored when an HF_TOKEN is present; that repo = same text tensors
  + a 267-tensor vision tower (vision.*, aligner.*, image_*) which we
  STRIP by name — those shards' vision tensors are simply skipped, so
  the vision weights are never even materialized.

Expert weights stay FP4 on host: expert_host[(layer_or_"mtp", e)] =
{"w1": u8 [I, D/2], "w1_s": u8 [I, D/32] e8m0, "w3", "w3_s",
 "w2": u8 [D, I/2], "w2_s"} — packed along K, low nibble first
(research/dsv4-port-spec.md §3).

Tensor name map (top-level, verified vs research/dsv4-index.json):
  embed.weight, head.weight, norm.weight, hc_head_{fn,base,scale}
  layers.L.attn.{wq_a,wq_b,wkv,wo_a,wo_b}.{weight,scale} (fp8 128x128)
  layers.L.attn.{q_norm,kv_norm}.weight, attn_sink (f32)
  layers.L.attn.compressor.{wkv,wgate}.weight (bf16), ape (f32),
      norm.weight
  layers.L.attn.indexer.{wq_b.weight,wq_b.scale}, weights_proj.weight
  layers.L.attn.indexer.compressor.{wkv,wgate,ape,norm}
  layers.L.{attn_norm,ffn_norm}.weight
  layers.L.ffn.gate.weight, gate.bias (f32) | gate.tid2eid (i32)
  layers.L.ffn.shared_experts.{w1,w2,w3}.{weight,scale} (fp8)
  layers.L.ffn.experts.E.{w1,w2,w3}.weight (fp4 u8) + .scale (e8m0 u8)
  mtp.0.* (same + e_proj/h_proj/enorm/hnorm/norm + own hc_head_*)
"""
from __future__ import annotations

import json
import os
import struct
import time
import urllib.request

import numpy as np

from .dsv4_config import Dsv4Config
from .fp8 import dequant_np

REPO = os.environ.get("DSV4_REPO", "deepseek-ai/DeepSeek-V4-Flash")

# the vision tower (267 tensors) — stripped, never downloaded materialized
STRIP_PREFIXES = ("vision.", "aligner.", "image_")


def _hdrs(token=None):
    h = {"User-Agent": "dsv4-tpu-kernel/1.0"}
    if token:
        h["Authorization"] = f"Bearer {token}"
    return h


def shard_header(url, token=None):
    req = urllib.request.Request(url, headers={**_hdrs(token),
                                               "Range": "bytes=0-7"})
    with urllib.request.urlopen(req, timeout=120) as r:
        hlen = struct.unpack("<Q", r.read())[0]
    req = urllib.request.Request(url, headers={**_hdrs(token),
                                               "Range": f"bytes=8-{8+hlen-1}"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.loads(r.read()), 8 + hlen


def _download_shard(url, out_path, token=None, n_conn=24):
    import concurrent.futures as cf
    import subprocess
    req = urllib.request.Request(url, headers={**_hdrs(token),
                                               "Range": "bytes=0-0"})
    with urllib.request.urlopen(req, timeout=120) as r:
        total = int(r.headers["Content-Range"].split("/")[-1])
    if not (os.path.exists(out_path) and os.path.getsize(out_path) == total):
        with open(out_path, "wb") as f:
            f.truncate(total)
    buf = np.memmap(out_path, dtype=np.uint8, mode="r+")
    stripe = max(16 << 20, total // n_conn)
    t0 = time.time()

    def fetch(rng):
        s, e = rng
        part = f"{out_path}.part{s}"
        cmd = ["curl", "-sL", "--fail", "--retry", "4", "-r", f"{s}-{e}"]
        for k, v in _hdrs(token).items():
            cmd += ["-H", f"{k}: {v}"]
        cmd += ["-o", part, url]
        subprocess.run(cmd, check=True)
        with open(part, "rb") as f:
            buf[s:e + 1] = np.frombuffer(f.read(), dtype=np.uint8)
        os.remove(part)

    with cf.ThreadPoolExecutor(max_workers=min(n_conn, 32)) as ex:
        list(ex.map(fetch, [(s, min(s + stripe - 1, total - 1))
                            for s in range(0, total, stripe)]))
    return buf, total, time.time() - t0


class _Views:
    def __init__(self, buf, ds):
        self.buf, self.ds = buf, ds

    def raw(self, meta):
        s, e = meta["data_offsets"]
        return self.buf[self.ds + s:self.ds + e]

    def u8(self, meta):
        return self.raw(meta).reshape(meta["shape"])

    def f32(self, meta):
        return np.frombuffer(self.raw(meta).tobytes(),
                             np.float32).reshape(meta["shape"])

    def bf16(self, meta):
        raw = self.raw(meta)
        arr = (raw.view(np.uint16).reshape(meta["shape"])
               if raw.flags["C_CONTIGUOUS"]
               else np.frombuffer(raw, np.uint16).reshape(meta["shape"]))
        return (arr.astype(np.uint32) << 16).view(np.float32)


def _ceil(n, d):
    return -(-n // d)


def _slice_rows(w, d, c):
    chunk = _ceil(w.shape[0], d)
    return w[c * chunk:(c + 1) * chunk]


def _slice_cols(w, d, c):
    chunk = _ceil(w.shape[1], d)
    return w[:, c * chunk:(c + 1) * chunk]


def carve_shard(header, views, cfg: Dsv4Config, d, chips, expert_host,
                out):
    """Carve all tensors of one shard into the engine layout."""

    def put(l, key, w, col=False, site="attn"):
        for c in range(d):
            tgt = chips[c][l].setdefault(site, {})
            tgt[key] = _slice_cols(w, d, c) if col else _slice_rows(w, d, c)

    # pre-pair fp8 weights with scales
    pairs = {}
    for name, meta in header.items():
        if name == "__metadata__" or not name.endswith(".scale"):
            continue
        wname = name[:-len(".scale")] + ".weight"
        if wname in header:
            pairs[wname] = meta

    for name, meta in header.items():
        if name == "__metadata__" or name.endswith(".scale"):
            continue
        if name.startswith(STRIP_PREFIXES):
            continue                                   # vision: stripped

        # ---- global ----
        if name == "embed.weight":
            out["embed"] = views.bf16(meta)
            continue
        if name == "head.weight":
            out["lm_head"] = views.bf16(meta)
            continue
        if name == "norm.weight":
            out["final_ln"] = views.bf16(meta)
            continue
        if name.startswith("hc_head_"):
            out.setdefault("head", {})[
                name[len("hc_head_"):]] = views.f32(meta)
            continue
        if not name.startswith(("layers.", "mtp.")):
            continue

        # ---- layer or mtp ----
        is_mtp = name.startswith("mtp.")
        if is_mtp:
            l, tail = "mtp", name[len("mtp.0."):]
            site_lay = chips[0]["mtp"]
        else:
            parts = name.split(".")
            l = int(parts[1])
            if l >= cfg.n_layers:
                continue
            tail = ".".join(parts[2:])
        lay = chips[0][l] if not is_mtp else site_lay

        def to_all(key, val, site):
            for c in range(d):
                chips[c]["mtp" if is_mtp else l].setdefault(
                    site, {})[key] = val

        # ---- hc + norms ----
        if tail.startswith("hc_attn_") or tail.startswith("hc_ffn_"):
            site = "attn_hc" if tail.startswith("hc_attn") else "ffn_hc"
            key = tail.split("_", 2)[2]
            to_all(key, views.f32(meta) if key != "fn"
                   else views.bf16(meta).astype(np.float32), site)
        elif tail == "attn_norm.weight":
            to_all("attn_norm", views.bf16(meta), "attn")
        elif tail == "ffn_norm.weight":
            to_all("ffn_norm", views.bf16(meta), "ffn")
        elif tail in ("enorm.weight", "hnorm.weight", "norm.weight") \
                and is_mtp:
            to_all({"enorm.weight": "enorm", "hnorm.weight": "hnorm",
                    "norm.weight": "norm"}[tail],
                   views.bf16(meta), "attn")

        # ---- attention weights ----
        elif tail.startswith("attn."):
            short = tail[len("attn."):]
            if short == "attn_sink":
                # f32 [n_heads] -> head-shard
                w = views.f32(meta)
                for c in range(d):
                    chips[c]["mtp" if is_mtp else l].setdefault(
                        "attn", {})["attn_sink"] = _slice_rows(w, d, c)
            elif short == "q_norm.weight":
                to_all("q_norm", views.bf16(meta), "attn")
            elif short == "kv_norm.weight":
                to_all("kv_norm", views.bf16(meta), "attn")
            elif short in ("wq_a.weight", "wq_b.weight", "wkv.weight",
                           "wo_a.weight", "wo_b.weight"):
                key = short[:-len(".weight")]
                smeta = pairs.get(name)
                if smeta is not None:
                    w = dequant_np(views.u8(meta),
                                   views.f32(smeta)).astype(np.float32)
                else:
                    w = views.bf16(meta)
                col = key == "wo_b"      # wo_b: col-shard over olr
                if is_mtp:
                    for c in range(d):
                        chips[c]["mtp"].setdefault("attn", {})[key] = \
                            _slice_cols(w, d, c) if col else \
                            _slice_rows(w, d, c)
                else:
                    put(l, key, w, col=col)

        # ---- compressor + indexer ----
        elif tail.startswith("attn.compressor.") or \
                tail.startswith("attn.indexer.compressor."):
            is_idx = "indexer" in tail
            site = "icomp" if is_idx else "comp"
            short = tail.split("compressor.")[-1]
            if short in ("wkv.weight", "wgate.weight"):
                to_all(short[:-len(".weight")], views.bf16(meta), site)
            elif short == "ape":
                to_all("ape", views.f32(meta), site)
            elif short == "norm.weight":
                to_all("norm", views.bf16(meta), site)
        elif tail.startswith("attn.indexer."):
            short = tail[len("attn.indexer."):]
            if short == "wq_b.weight":
                smeta = pairs.get(name)
                w = dequant_np(views.u8(meta),
                               views.f32(smeta)).astype(np.float32)
                to_all("wq_b", w, "idx")
            elif short == "weights_proj.weight":
                to_all("weights_proj", views.bf16(meta), "idx")

        # ---- FFN ----
        elif tail.startswith("ffn."):
            if tail == "ffn.gate.weight":
                to_all("w", views.bf16(meta), "gate")
            elif tail == "ffn.gate.bias":
                to_all("bias", views.f32(meta), "gate")
            elif tail == "ffn.gate.tid2eid":
                raw = views.raw(meta)
                arr = (raw.view(np.int32).reshape(meta["shape"])
                       if isinstance(raw, np.ndarray)
                       else np.frombuffer(raw, np.int32).reshape(
                           meta["shape"]))
                to_all("tid2eid", arr, "gate")
            elif tail.startswith("ffn.shared_experts."):
                short = tail[len("ffn.shared_experts."):]
                if short.endswith(".weight"):
                    key = short[:-len(".weight")]
                    w = dequant_np(views.u8(meta),
                                   views.f32(pairs[name])).astype(np.float32)
                    for c in range(d):
                        tgt = chips[c]["mtp" if is_mtp else
                                      l].setdefault("shared", {})
                        tgt[key] = _slice_cols(w, d, c) if key == "w2" \
                            else _slice_rows(w, d, c)
            elif tail.startswith("ffn.experts."):
                # experts.E.{w1,w2,w3}.weight (fp4 packed) + .scale (e8m0)
                parts = tail[len("ffn.experts."):].split(".")
                e, base = int(parts[0]), parts[1]
                if base not in ("w1", "w2", "w3"):
                    continue
                key = ("mtp", e) if is_mtp else (l, e)
                ex = expert_host.setdefault(key, {})
                ex[base] = views.u8(meta)             # packed fp4 bytes
                smeta = pairs.get(name)
                if smeta is not None:
                    ex[base + "_s"] = views.u8(smeta)  # e8m0 u8 scales

        # ---- MTP projections ----
        elif is_mtp and tail in ("e_proj.weight", "h_proj.weight"):
            key = tail[:-len(".weight")]
            smeta = pairs.get(name)
            if smeta is not None:
                w = dequant_np(views.u8(meta),
                               views.f32(smeta)).astype(np.float32)
            else:
                w = views.bf16(meta)
            to_all(key, w, "attn")


def finalize_shards(chips, cfg, d, log=print):
    """Convert bf16-valued params to f32 containers and fill structural
    defaults for layers/tails not present in every repo variant."""
    for c in range(d):
        for l in range(cfg.n_layers):
            lay = chips[c][l]
            # gate bias default (hash layers have none)
            g = lay.setdefault("gate", {})
            g.setdefault("bias", None)
            g.setdefault("tid2eid", None)
            # comp/idx/icomp defaults
            lay.setdefault("comp", None)
            lay.setdefault("idx", None)
            lay.setdefault("icomp", None)
        m = chips[c]["mtp"]
        m.setdefault("gate", {}).setdefault("bias", None)
        m["gate"].setdefault("tid2eid", None)


def load_real(cfg: Dsv4Config, d: int, token=None, log=print,
              workdir="/dev/shm/dsv4w", max_shards=None):
    """Full load: returns (params_by_chip, embed, lm_head, expert_host)."""
    if token is None:
        token = os.environ.get("HF_TOKEN") or None
    os.makedirs(workdir, exist_ok=True)
    idx_url = f"https://huggingface.co/{REPO}/resolve/main/" \
              "model.safetensors.index.json"
    with urllib.request.urlopen(
            urllib.request.Request(idx_url, headers=_hdrs(token)),
            timeout=120) as r:
        index = json.loads(r.read())
    files = sorted(set(index["weight_map"].values()))
    if max_shards:
        files = files[:max_shards]
    log(f"[load] repo {REPO}: {len(files)} shards to fetch")

    D = cfg.hidden_size
    chips = [{l: {"attn": {}, "ffn_hc": {}, "attn_hc": {}, "ffn": {},
                  "gate": {}, "shared": {}, "comp": {}, "idx": {},
                  "icomp": {}}
              for l in range(cfg.n_layers)} for _ in range(d)]
    for c in range(d):
        chips[c]["mtp"] = {"attn": {}, "attn_hc": {}, "ffn_hc": {},
                           "ffn": {}, "gate": {}, "shared": {}}
    expert_host = {}
    out = {}

    t0 = time.time()
    n_seen = 0
    for fi, fname in enumerate(files):
        url = f"https://huggingface.co/{REPO}/resolve/main/{fname}"
        out_path = os.path.join(workdir, fname)
        buf, total, dt = _download_shard(url, out_path, token)
        header, ds = shard_header(url, token)
        carve_shard(header, _Views(buf, ds), cfg, d, chips, expert_host,
                    out)
        n_seen += 1
        log(f"[load] {fi+1}/{len(files)} {fname} "
            f"({total/1e9:.1f} GB, {dt:.0f}s)")
    finalize_shards(chips, cfg, d, log=log)
    log(f"[load] done in {(time.time()-t0)/60:.1f} min; "
        f"experts: {len(expert_host)}")

    embed = out["embed"]
    lm_head = out["lm_head"]
    for c in range(d):
        chips[c]["head"] = out["head"]
        chips[c]["final_ln"] = out["final_ln"]
    return chips, embed, lm_head, expert_host
'''
    MODS['dsv4_chat.py'] = '''"""DeepSeek-V4 chat encoding — port of DeepSeek's encoding/encoding_dsv4.py
(research/dsv4-encoding.py, the repo's official chat template; the HF repo
ships no chat_template.jinja).  Implements the subset needed for serving:
encode_messages (thinking_mode chat/thinking) and completion parsing that
splits reasoning (<think> blocks) for reasoning_content.
"""
from __future__ import annotations

from typing import Any, Dict, List

bos_token = "<｜begin▁of▁sentence｜>"
eos_token = "<｜end▁of▁sentence｜>"
thinking_start_token = "<think>"
thinking_end_token = "</think>"
dsml_token = "｜DSML｜"

USER_SP_TOKEN = "<｜User｜>"
ASSISTANT_SP_TOKEN = "<｜Assistant｜>"
LATEST_REMINDER_SP_TOKEN = "<｜latest_reminder｜>"

assistant_msg_template = "{reasoning}{content}{tool_calls}" + eos_token
assistant_msg_wo_eos_template = "{reasoning}{content}{tool_calls}"
thinking_template = "{reasoning_content}"


def _has_image_or_video(messages) -> bool:
    for m in messages:
        c = m.get("content")
        if isinstance(c, list):
            for part in c:
                if isinstance(part, dict) and part.get("type") in (
                        "image_url", "image", "video", "video_url",
                        "input_image", "input_video"):
                    return True
    return False


def find_last_user_index(messages) -> int:
    for i in range(len(messages) - 1, -1, -1):
        if messages[i].get("role") == "user":
            return i
    return -1


def _content_str(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for p in content:
            if isinstance(p, dict) and p.get("type") == "text":
                parts.append(p.get("text", ""))
        return "\n".join(parts)
    return ""


def render_message(index, messages, thinking_mode, drop_thinking):
    msg = messages[index]
    role = msg.get("role")
    content = _content_str(msg.get("content", ""))

    if role == "system":
        prompt = content
    elif role in ("user", "developer", "latest_reminder"):
        prompt = content
        # tool_result blocks
        blocks = msg.get("content_blocks") or []
        tr = [b.get("content", "") for b in blocks
              if b.get("type") == "tool_result"]
        if tr:
            prompt += "".join(f"<tool_result>{t}</tool_result>" for t in tr)
    elif role == "assistant":
        rc = msg.get("reasoning_content") or ""
        thinking_part = ""
        last_user_idx = find_last_user_index(messages)
        if thinking_mode == "thinking" and not (
                drop_thinking and index < last_user_idx):
            thinking_part = thinking_template.format(
                reasoning_content=rc) + thinking_end_token
        if msg.get("wo_eos"):
            prompt = assistant_msg_wo_eos_template.format(
                reasoning=thinking_part, content=content, tool_calls="")
        else:
            prompt = assistant_msg_template.format(
                reasoning=thinking_part, content=content, tool_calls="")
    else:
        raise NotImplementedError(f"Unknown role: {role}")

    # transition tokens after the final message
    if index + 1 < len(messages) and messages[index + 1].get("role") \
            not in ["assistant", "latest_reminder"]:
        return prompt
    if messages[index].get("role") in ("user", "developer"):
        prompt += ASSISTANT_SP_TOKEN
        if not drop_thinking and thinking_mode == "thinking":
            prompt += thinking_start_token
        elif drop_thinking and thinking_mode == "thinking" \
                and index >= find_last_user_index(messages):
            prompt += thinking_start_token
        else:
            prompt += thinking_end_token
    return prompt


def encode_messages(messages: List[Dict[str, Any]], thinking_mode="chat",
                    drop_thinking=True, add_default_bos_token=True) -> str:
    """Main entry: OpenAI-style messages -> DSV4 prompt string."""
    prompt = bos_token if add_default_bos_token else ""
    last_user_idx = find_last_user_index(messages)
    for idx in range(len(messages)):
        prompt += render_message(idx, messages, thinking_mode,
                                 drop_thinking)
    return prompt


def parse_message_from_completion_text(text: str,
                                       thinking_mode="chat") -> Dict[str, Any]:
    """Split a completion into reasoning_content / content (DSV4 thinks:
    output starts inside a <think> block when thinking_mode='thinking')."""
    msg = {"role": "assistant", "content": "", "reasoning_content": ""}
    if thinking_mode == "thinking":
        if text.startswith(thinking_start_token):
            # everything up to </think> is reasoning
            rest = text[len(thinking_start_token):]
            if thinking_end_token in rest:
                rc, content = rest.split(thinking_end_token, 1)
                msg["reasoning_content"] = rc
                msg["content"] = content
                return msg
            msg["reasoning_content"] = rest
            return msg
        if thinking_end_token in text:
            rc, content = text.split(thinking_end_token, 1)
            msg["reasoning_content"] = rc
            msg["content"] = content
            return msg
    msg["content"] = text
    return msg
'''
    MODS['dsv4_glue.py'] = '''"""ModelRunner glue for the DSV4 engine: tokenizer + chat encoding +
server-side context auto-compaction.

Auto-compaction (spec: prevents platform OOM at long conversations):
when the rendered conversation crosses 85% of max_ctx, the OLDEST turns
(keep a fixed watermark of the most recent messages) are summarized by
the model itself (greedy = deterministic, chunked at 2000 tokens) and
replaced with a single system summary message.  The client never sees
it: the runner stores compacted history internally and serves from it.
"""
from __future__ import annotations

import time

import numpy as np

from . import dsv4_chat
from .dsv4_config import Dsv4Config
from .dsv4_runtime import Dsv4Runner

COMPACT_FRAC = 0.85          # trigger threshold
COMPACT_KEEP = 4             # watermark: messages kept verbatim
COMPACT_CHUNK = 2000         # tokens per summarization chunk
COMPACT_MAX_OUT = 200        # summary length cap (tokens)


class Dsv4ModelRunner:
    """openai_api.ModelRunner implementation over Dsv4Runner."""

    def __init__(self, runner: Dsv4Runner, tokenizer,
                 thinking_mode: str = "chat", log=print):
        self.r = runner
        self.tok = tokenizer
        self.thinking_mode = thinking_mode
        self.log = log
        self.eos = set(runner.cfg.eos_ids)
        self.messages = []         # compacted conversation state
        self.compactions = 0

    # ------------------------------------------------------------------
    def _encode(self, messages):
        text = dsv4_chat.encode_messages(
            messages, thinking_mode=self.thinking_mode,
            drop_thinking=True)
        enc = self.tok.encode(text, add_special_tokens=False)
        ids = list(enc.ids) if hasattr(enc, "ids") else list(enc)
        return ids

    def _n_tokens(self, messages):
        return len(self._encode(messages))

    # ------------------------------------------------------------------
    def _summarize(self, text: str, max_out=None) -> str:
        """Deterministic greedy summarization via the model itself.
        Chunked: long inputs are summarized piecewise, then combined."""
        max_out = max_out or COMPACT_MAX_OUT
        chunks = []
        toks = self.tok.encode(text, add_special_tokens=False)
        toks = list(toks.ids) if hasattr(toks, "ids") else list(toks)
        pieces = [toks[i:i + COMPACT_CHUNK]
                  for i in range(0, len(toks), COMPACT_CHUNK)]
        for piece in pieces:
            part = self.tok.decode(piece)
            msgs = [{"role": "user",
                     "content": "Summarize the following conversation "
                     "turns in under "
                     f"{max_out} tokens, preserving key facts, decisions "
                     "and open tasks:\n\n" + part}]
            out = self._gen(msgs, max_tokens=max_out, temperature=0.0)
            chunks.append(out.strip())
        return "\n".join(chunks)[-max_out * 4:]     # hard char cap

    def _gen(self, messages, max_tokens=512, temperature=0.0, top_p=1.0,
             stream_cb=None):
        ids = self._encode(messages)
        ids = ids[-(self.r.cfg.max_ctx - max_tokens - 8):]
        logits = self.r.prefill(ids)
        out_ids = []
        det_text = ""
        for i in range(max_tokens):
            t = self.r._sample(logits, temperature, top_p)
            if t in self.eos:
                break
            out_ids.append(t)
            if stream_cb and (i + 1) % 4 == 0:
                piece = self.tok.decode(out_ids)
                if piece != det_text:
                    stream_cb(piece[len(det_text):], i)
                    det_text = piece
            if i + 1 < max_tokens:
                logits, _ = self.r._decode_one(t, temperature, top_p)
        return self.tok.decode(out_ids)

    # ------------------------------------------------------------------
    def _maybe_compact(self):
        """Server-side auto-compaction: replace oldest turns with a model
        summary when the conversation crosses 85% of max_ctx.  Iterative:
        each pass shrinks the kept history and the summary budget until
        the rendered conversation fits."""
        for _ in range(8):
            n = self._n_tokens(self.messages)
            limit = int(self.r.cfg.max_ctx * COMPACT_FRAC)
            if n <= limit or len(self.messages) <= 1:
                return
            keep_n = min(COMPACT_KEEP, len(self.messages) - 1)
            keep = self.messages[-keep_n:] if keep_n else []
            # shrink the watermark until the kept tail alone fits
            while keep_n > 1 and self._n_tokens(keep) > limit - 64:
                keep_n -= 1
                keep = self.messages[-keep_n:]
            old = self.messages[:-keep_n] if keep_n else self.messages
            lines = []
            for m in old:
                c = m.get("content", "")
                lines.append(f"{m.get('role', 'user')}: {c}")
            text = "\n".join(lines)
            budget = max(16, limit - self._n_tokens(keep) - 64)
            summary = self._summarize(text, max_out=budget)
            head = ("[Conversation summary (auto-compacted by the server "
                    "to stay within the context window)]\n")
            new_msgs = [{"role": "system", "content": head + summary}] + keep
            # hard fit check: shrink the summary until it fits
            while self._n_tokens(new_msgs) > limit and len(summary) > 16:
                summary = summary[:max(16, len(summary) // 2)]
                new_msgs = [{"role": "system",
                             "content": head + summary}] + keep
            self.messages = new_msgs
            self.compactions += 1
            self.log(f"[compact] {n} tok > {limit} ({COMPACT_FRAC:.0%} "
                     f"of max_ctx) -> summarized {len(old)} oldest "
                     f"messages ({self._n_tokens(self.messages)} tok "
                     f"now, compaction #{self.compactions})")

    # ------------------------------------------------------------------
    def chat(self, messages, max_tokens=512, temperature=0.0, top_p=1.0,
             stream_cb=None, reasoning_effort=None):
        if dsv4_chat._has_image_or_video(messages):
            raise ValueError(
                "image/video inputs are not supported by this engine "
                "(vision tower weights are stripped at load; text-only "
                "serving)")
        # append user turn to the compacted history
        self.messages = self.messages + [dict(m) for m in messages
                                         if m.get("role") == "user"] \
            if not self._is_full_conversation(messages) else \
            [dict(m) for m in messages]
        self._maybe_compact()
        t0 = time.time()
        text = self._gen(self.messages, max_tokens=max_tokens,
                         temperature=temperature, top_p=top_p,
                         stream_cb=stream_cb)
        # record the assistant reply in history
        parsed = dsv4_chat.parse_message_from_completion_text(
            text, self.thinking_mode)
        self.messages.append(parsed)
        dt = time.time() - t0
        self.log(f"[gen] {len(text)} chars in {dt:.1f}s")
        return text

    @staticmethod
    def _is_full_conversation(messages):
        # clients that send the whole history each call (stateless use)
        return len(messages) > 1

    def reset(self):
        self.messages = []
        self.compactions = 0
        self.r.reset()
'''
    MODS['dsv4_openai.py'] = '''"""OpenAI-compatible API server for the DSV4 engine (stdlib only).

Differences from the GLM server (openai_api.py):
  - reasoning_content split: DSV4 thinks — streaming emits
    delta.reasoning_content while inside <think>…</think>, then
    delta.content afterwards (DeepSeek style).
  - clean 400 error on image/video content parts.
  - model id deepseek-v4-flash (or -vision-uncensored variant).
"""
from __future__ import annotations

import json
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

from . import dsv4_chat

API_KEY = "kaggle-sfw-token-9999"
HOST = "0.0.0.0"
PORT = 8080
MODEL_ID = "deepseek-v4-flash"

_model = None
_lock = threading.Lock()


class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, fmt, *args):
        pass

    def _auth(self):
        h = self.headers.get("Authorization", "")
        return h == f"Bearer {API_KEY}"

    def _send_json(self, code, obj):
        body = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path == "/healthz":
            self._send_json(200, {"ok": True})
        elif self.path == "/v1/models":
            self._send_json(200, {"object": "list", "data": [
                {"id": MODEL_ID, "object": "model",
                 "owned_by": "local"}]})
        else:
            self._send_json(404, {"error": "not found"})

    def do_POST(self):
        if not self._auth():
            self._send_json(401, {"error": "bad api key"})
            return
        n = int(self.headers.get("Content-Length", 0))
        body = json.loads(self.rfile.read(n) or b"{}")
        if self.path == "/v1/chat/completions":
            self._chat(body)
        else:
            self._send_json(404, {"error": "not found"})

    def _chat(self, body):
        messages = body.get("messages", [])
        if dsv4_chat._has_image_or_video(messages):
            self._send_json(400, {"error": {
                "message": "Image and video inputs are not supported by "
                           "this engine (vision tower weights are "
                           "stripped; text-only serving).",
                "type": "invalid_request_error", "code": "modalitiy_not_supported"}})
            return
        max_tokens = int(body.get("max_tokens", 512))
        temperature = float(body.get("temperature", 0.0))
        top_p = float(body.get("top_p", 1.0))
        stream = bool(body.get("stream", False))
        t0 = time.time()
        with _lock:
            if stream:
                self._stream_chat(messages, max_tokens, temperature, top_p)
                return
            try:
                text = _model.chat(messages, max_tokens=max_tokens,
                                   temperature=temperature, top_p=top_p)
            except ValueError as e:
                self._send_json(400, {"error": {"message": str(e),
                                                "type": "invalid_request_error"}})
                return
        parsed = dsv4_chat.parse_message_from_completion_text(
            text, _model.thinking_mode)
        self._send_json(200, {
            "id": "chatcmpl-local", "object": "chat.completion",
            "created": int(time.time()), "model": MODEL_ID,
            "choices": [{"index": 0, "finish_reason": "stop",
                         "message": {
                             "role": "assistant",
                             "content": parsed["content"],
                             "reasoning_content":
                                 parsed["reasoning_content"] or None}}],
            "usage": {"prompt_tokens": 0,
                      "completion_tokens": len(text.split()),
                      "total_tokens": len(text.split())},
            "_timing_s": round(time.time() - t0, 2),
        })

    def _stream_chat(self, messages, max_tokens, temperature, top_p):
        self.send_response(200)
        self.send_header("Content-Type", "text/event-stream")
        self.send_header("Cache-Control", "no-cache")
        self.end_headers()
        state = {"in_think": True, "started_think": False}

        def cb(tok_text, idx):
            # split reasoning vs content by </think>
            nonlocal state
            pieces = []
            buf = tok_text
            while buf:
                if state["in_think"]:
                    if not state["started_think"]:
                        state["started_think"] = True
                    if "</think>" in buf:
                        pre, buf = buf.split("</think>", 1)
                        if pre:
                            pieces.append(("reasoning", pre))
                        state["in_think"] = False
                    else:
                        pieces.append(("reasoning", buf))
                        buf = ""
                else:
                    pieces.append(("content", buf))
                    buf = ""
            for kind, txt in pieces:
                if not txt:
                    continue
                delta = {"reasoning_content": txt} if kind == "reasoning" \
                    else {"content": txt}
                chunk = {"id": "chatcmpl-local",
                         "object": "chat.completion.chunk",
                         "created": int(time.time()), "model": MODEL_ID,
                         "choices": [{"index": 0, "delta": delta}]}
                try:
                    self.wfile.write(f"data: {json.dumps(chunk)}\n\n".encode())
                    self.wfile.flush()
                except Exception:
                    pass

        try:
            _model.chat(messages, max_tokens=max_tokens,
                        temperature=temperature, top_p=top_p,
                        stream_cb=cb)
        except Exception:
            pass
        done = {"id": "chatcmpl-local", "object": "chat.completion.chunk",
                "choices": [{"index": 0, "delta": {},
                             "finish_reason": "stop"}]}
        try:
            self.wfile.write(f"data: {json.dumps(done)}\n\n".encode())
            self.wfile.write(b"data: [DONE]\n\n")
            self.wfile.flush()
        except Exception:
            pass


def serve(model, port=None, host=None, model_id=None):
    global _model, PORT, HOST, MODEL_ID
    _model = model
    if port:
        PORT = port
    if host:
        HOST = host
    if model_id:
        MODEL_ID = model_id
    httpd = ThreadingHTTPServer((HOST, PORT), Handler)
    print(f"[server] listening on http://{HOST}:{PORT}/v1  "
          f"(key: {API_KEY[:8]}...)", flush=True)
    httpd.serve_forever()
'''
    MODS['fp8.py'] = '''"""FP8 (e4m3fn) 128x128-blockwise dequantization.

numpy LUT for host-side dequant (dense/small weights -> BF16 at load time);
JAX in-graph version only for routed-expert banks (whole experts, row0=col0=0).
"""
from __future__ import annotations

import numpy as np

BLOCK = 128


def _build_lut() -> np.ndarray:
    lut = np.zeros(256, dtype=np.float32)
    for u in range(256):
        sign = -1.0 if (u & 0x80) else 1.0
        exp = (u >> 3) & 0xF
        man = u & 0x7
        if exp == 0:
            val = (2.0 ** -6) * (man / 8.0)          # subnormal
        elif (u & 0x7F) == 0x7F:
            val = 0.0                                 # NaN -> 0 (not in weights)
        else:
            val = (2.0 ** (exp - 7)) * (1.0 + man / 8.0)
        lut[u] = sign * val
    return lut


F8_LUT = _build_lut()


def dequant_np(fp8_u8: np.ndarray, scale_inv: np.ndarray) -> np.ndarray:
    """uint8 [R,C] + f32 scale_inv [ceil(R/128), ceil(C/128)] -> f32 [R,C].
    W = fp8 * scale_inv (scale_inv holds the INVERSE scale)."""
    R, C = fp8_u8.shape
    vals = F8_LUT[fp8_u8]                               # [R,C] f32 LUT gather
    out = np.empty((R, C), dtype=np.float32)
    for i in range(0, R, BLOCK):
        for j in range(0, C, BLOCK):
            out[i:i + BLOCK, j:j + BLOCK] = (
                vals[i:i + BLOCK, j:j + BLOCK] * scale_inv[i // BLOCK, j // BLOCK])
    return out


def fp8_matmul(x, fp8_u8, scale_inv, block=BLOCK):
    """JAX in-graph: x [.., in] bf16 @ dequant(fp8 [out, in])^T -> [.., out].
    Only used for whole-expert banks (global row/col both start at 0)."""
    import jax.numpy as jnp
    from jax import lax
    out_dim, in_dim = fp8_u8.shape
    fp8 = lax.bitcast_convert_type(fp8_u8, jnp.float8_e4m3fn).astype(jnp.float32)
    row = jnp.arange(out_dim) // block
    col = jnp.arange(in_dim) // block
    s = scale_inv[row[:, None], col[None, :]]           # [out, in] f32
    w = (fp8 * s).astype(jnp.bfloat16)
    return x @ w.T
'''
    MODS['__init__.py'] = ''''''
    os.makedirs("/kaggle/tmp/glmtpu", exist_ok=True)
    for name, src in MODS.items():
        with open(f"/kaggle/tmp/glmtpu/{name}", "w", encoding="utf-8") as f:
            f.write(src)
    sys.path.insert(0, "/kaggle/tmp")

from glmtpu.dsv4_config import Dsv4Config
import glmtpu.dsv4_runtime, glmtpu.dsv4_loader
print("engine modules ready; loader repo =",
      glmtpu.dsv4_loader.REPO)

In [ ]:
# ---- engine self-test on the real TPU (tiny cfg, fake weights) ----
import numpy as np
from glmtpu.dsv4_config import Dsv4Config
from glmtpu.dsv4_params import make_fake
from glmtpu.dsv4_runtime import Dsv4Runner

cfg = Dsv4Config.tiny()
pbc, embed, lm_head, expert_host = make_fake(cfg, d=8, seed=1)
r = Dsv4Runner(cfg, pbc, embed, lm_head, expert_host)
tokens = np.random.randint(0, cfg.vocab_size, size=40).tolist()
g = r.generate(tokens, max_new_tokens=8, temperature=0.0)
g2 = r.generate(tokens, max_new_tokens=8, temperature=0.0)
assert g == g2, "greedy determinism failed"
stats = {}
gm = r.generate_mtp(tokens, max_new_tokens=8, temperature=0.0, stats=stats)
assert gm == g, "MTP-1 greedy must be lossless"
print("TPU self-test OK:", g, "| mtp stats:", stats)

In [ ]:
# ---- tokenizer (DeepSeek-V4) ----
import urllib.request

BASE = "https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash/resolve/main"
for fn in ["tokenizer.json", "tokenizer_config.json"]:
    urllib.request.urlretrieve(f"{BASE}/{fn}", f"/kaggle/tmp/{fn}")
    print("got", fn)

from tokenizers import Tokenizer
tok = Tokenizer.from_file("/kaggle/tmp/tokenizer.json")
print("vocab:", tok.get_vocab_size())

In [ ]:
# ---- load real weights: 46 shards -> host RAM (experts stay FP4) ----
import numpy as np
from glmtpu.dsv4_config import Dsv4Config
from glmtpu.dsv4_loader import load_real
from glmtpu.dsv4_runtime import Dsv4Runner

t0 = time.time()
cfg = Dsv4Config.real(max_ctx=262144)   # 256k default context
cfg.n_slots = 8                         # hot fp4 experts per chip/layer
params_by_chip, embed, lm_head, expert_host = load_real(
    cfg, d=8, log=print, workdir="/dev/shm/dsv4w")
print(f"weights loaded in {(time.time()-t0)/60:.1f} min; "
      f"experts: {len(expert_host)}")

runner = Dsv4Runner(cfg, params_by_chip, embed, lm_head, expert_host,
                    log=print)
print("runner ready")
del params_by_chip

In [ ]:
# ---- generation sanity check (chat encoding, thinking mode) ----
from glmtpu import dsv4_chat

msgs = [{"role": "user", "content": "Say something with teeth. Be brief."}]
prompt = dsv4_chat.encode_messages(msgs, thinking_mode="thinking")
ids = tok.encode(prompt, add_special_tokens=False)
print("prompt tokens:", len(ids))
t0 = time.time()
logits = runner.prefill(ids.ids if hasattr(ids, "ids") else ids)
out = []
L = logits
for i in range(64):
    t = runner._sample(L, 0.7, 0.95)
    if t in runner.cfg.eos_ids:
        break
    out.append(t)
    L, _ = runner._decode_one(t, 0.7, 0.95)
text = tok.decode(out)
parsed = dsv4_chat.parse_message_from_completion_text(text, "thinking")
print(f"generated {len(out)} tokens in {time.time()-t0:.1f}s")
print("REASONING:", parsed["reasoning_content"][:200])
print("CONTENT:", parsed["content"][:300])

In [ ]:
# ---- OpenAI-compatible server + cloudflared tunnel (with compaction) ----
import threading, subprocess, re, time

from glmtpu import dsv4_openai
from glmtpu.dsv4_glue import Dsv4ModelRunner

model = Dsv4ModelRunner(runner, tok, thinking_mode="thinking")
server = threading.Thread(target=dsv4_openai.serve,
                          args=(model, 8080, "0.0.0.0"),
                          kwargs={"model_id": "deepseek-v4-flash"},
                          daemon=True)
server.start()
time.sleep(2)

cf = "/kaggle/tmp/cloudflared"
if not os.path.exists(cf):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", cf], check=True)
    subprocess.run(["chmod", "+x", cf], check=True)
subprocess.Popen([cf, "tunnel", "--url", "http://localhost:8080",
                  "--no-autoupdate"],
                 stdout=open("/kaggle/tmp/tunnel.log", "w"),
                 stderr=subprocess.STDOUT)
url = None
for _ in range(24):
    time.sleep(5)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                  open("/kaggle/tmp/tunnel.log").read())
    if m:
        url = m.group(0)
        break
assert url, "no tunnel URL"
print("=" * 60)
print("PUBLIC API:", url + "/v1")
print("API KEY:   kaggle-sfw-token-9999")
print("(server auto-compacts long conversations at 85% of 256k ctx)")
print("=" * 60)
with open("/kaggle/working/api_url.txt", "w") as f:
    f.write(url + "/v1\n")

In [ ]:
# ---- smoke test through the public URL (openai client) ----
import subprocess
subprocess.run(["pip", "install", "-q", "openai"], check=False)
import re
from openai import OpenAI
url = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                open("/kaggle/tmp/tunnel.log").read()).group(0)
c = OpenAI(base_url=f"{url}/v1", api_key="kaggle-sfw-token-9999")
r = c.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[{"role": "user", "content": "Say something with teeth."}],
    max_tokens=512, temperature=0.7, stream=True)
reasoning, content = "", ""
for chunk in r:
    d = chunk.choices[0].delta
    if getattr(d, "reasoning_content", None):
        reasoning += d.reasoning_content
    if getattr(d, "content", None):
        content += d.content
print("REASONING:", reasoning[:200])
print("CONTENT:", content[:400])
with open("/kaggle/working/api_smoke.txt", "w") as f:
    f.write(content or "")
# image requests get a clean 400:
try:
    c.chat.completions.create(
        model="deepseek-v4-flash",
        messages=[{"role": "user", "content": [
            {"type": "text", "text": "hi"},
            {"type": "image_url", "image_url": {"url": "http://x/x.png"}}]}],
        max_tokens=8)
    print("UNEXPECTED: image request accepted")
except Exception as e:
    print("image request correctly rejected:", str(e)[:120])

In [ ]:
# ---- keep alive until the session ends ----
import time
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    pass